# Student-Relative Reliability Gating for Risk-Controlled Distillation in Engineering Time-Series Forecasting

**Target venue: Results in Engineering (Elsevier) — v4 benchmark notebook.**

## Scope of this revision (v3 -> v4)
This notebook upgrades the v3 artifact to satisfy the pre-submission specification (R1-R15).
The persistence-relative gate of v3 is retained **only as a baseline**; the primary method is a
**student-relative, calibrated, reject-capable gate** (SR-gate) with an exact `g = 0` state.

| Blocker | v4 implementation |
|---|---|
| R4 gate redesign | `student_relative_reliability` + three-action policy (reject / attenuate / accept), no floor |
| R5 comparator suite | NS, NaiveKD, HardReject-SR, Soft-SR, Legacy-Persist (trained) + ValSelect, HardReject-Persist, Oracle (post-hoc) |
| R6 strong students | DLinear, PatchTST, N-HiTS as primary students; CeNN demoted to the mechanistic legacy study |
| R7 temporal validation | expanding rolling origins with non-overlapping test blocks and per-origin gate calibration |
| R8 inference | dataset x origin experimental units; Wilcoxon over origin blocks (primary), per-origin Diebold-Mariano with HAC variance and moving-block bootstrap (sensitivity); per-window tests demoted to descriptive |
| R13 objective | strong students use the simplified objective `L_pred + g * lambda * L_KD`; the CeNN ablation adds nested calibration-based configuration selection |
| R14 safety endpoints | negative-transfer rate, regret (mean / median / P95 / CVaR95), benefit retention, divergence, risk-coverage curve |
| R15 auditability | pre-registered thresholds dumped before training; checkpointed, resumable benchmark; per-block raw prediction-error shards |

Parts I-II (data, teachers, CeNN, single-split benchmark) are the **legacy study** kept for
the mechanistic section and the v3-comparison appendix. Part III-bis is the **primary benchmark**.

AAPL is retained **only as an out-of-domain stress test**; primary engineering claims are
restricted to the energy / climate / infrastructure series (R1).

## 1. Environment, determinism, IEEE figure style

In [1]:
import os, sys, json, time, gzip, zipfile, io, hashlib, platform, random, urllib.request, subprocess, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from scipy import stats
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings("ignore")

WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
RESULTS_DIR, FIG_DIR, DATA_DIR = (os.path.join(WORKDIR, d) for d in ("results", "figures", "data"))
for d in (RESULTS_DIR, FIG_DIR, DATA_DIR): os.makedirs(d, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"; os.environ["PYTHONHASHSEED"] = "0"
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
try: torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as e: print("Determinism note:", e)
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

# --- IEEE-compliant figure style: vector PDF (Type 42 fonts), serif, column widths.
FIG_W1, FIG_W2 = 3.5, 7.16   # single / double IEEE column width (inches)
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "pdf.fonttype": 42, "ps.fonttype": 42,           # embed TrueType (IEEE requirement)
    "font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "dejavuserif",
    "font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 6.5,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linewidth": 0.4, "axes.axisbelow": True,
    "axes.linewidth": 0.6, "lines.linewidth": 1.0,
})
TEACHER_COLORS = {"QELM": "#8172B3", "VQC": "#DD8452", "QRC": "#55A868", "QKRR": "#4C72B0"}
PALETTE = {"CeNN_NS": "#4C72B0", "Persistence": "#8C8C8C", "DLinear": "#CCB974",
           "NLinear": "#937860", "MLP": "#64B5CD", "N-BEATS": "#DA8BC3"}
def savefig(fig, name):
    fig.savefig(f"{FIG_DIR}/{name}.png", bbox_inches="tight")
    fig.savefig(f"{FIG_DIR}/{name}.pdf", bbox_inches="tight")   # vector PDF for the paper
    plt.close(fig); print(f"[fig] {name}.pdf / .png saved")

ENV = {"python": platform.python_version(), "platform": platform.platform(),
       "torch": torch.__version__, "numpy": np.__version__,
       "scipy": __import__("scipy").__version__, "pandas": pd.__version__,
       "device": DEVICE, "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None}
json.dump(ENV, open(f"{RESULTS_DIR}/environment.json", "w"), indent=2)
print(json.dumps(ENV, indent=2))
T_START = time.time()
def elapsed(): return f"{(time.time()-T_START)/60:.1f} min"

{
  "python": "3.12.13",
  "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
  "torch": "2.10.0+cu128",
  "numpy": "2.0.2",
  "scipy": "1.16.3",
  "pandas": "2.3.3",
  "device": "cuda",
  "gpu": "Tesla T4"
}


## 2. Configuration and pre-registration

In [2]:
RUN_MODE = "full"    # "smoke" | "fast" | "full"

WINDOW = 96                       # standard LTSF look-back
MAIN_HORIZON = 24
MULTI_HORIZONS = [24, 48, 96]
SPLIT_RATIOS = (0.70, 0.15, 0.15)

PROFILES = {
    "smoke": dict(max_points=3000,  seeds=[42],                 mh_seeds=[42],           abl_seeds=[42],
                  epochs=6,  patience=3,  batch_size=128, hidden_dim=48,  n_cells=48,
                  vqc_epochs=6,  qkrr_landmarks=128, multi_horizon=False, bootstrap=200),
    "fast":  dict(max_points=9000,  seeds=[42, 123, 2024],      mh_seeds=[42, 123],      abl_seeds=[42, 123],
                  epochs=35, patience=8,  batch_size=256, hidden_dim=96,  n_cells=96,
                  vqc_epochs=25, qkrr_landmarks=384, multi_horizon=True,  bootstrap=2000),
    "full":  dict(max_points=20000, seeds=[42, 123, 2024, 7, 99], mh_seeds=[42, 123, 2024], abl_seeds=[42, 123, 2024],
                  epochs=60, patience=10, batch_size=256, hidden_dim=128, n_cells=128,
                  vqc_epochs=30, qkrr_landmarks=512, multi_horizon=True,  bootstrap=5000),
}
CFG = dict(PROFILES[RUN_MODE]); CFG["window"] = WINDOW

TEACHERS = ["QELM", "VQC", "QRC", "QKRR"]        # the four QML families under study

LOSS_WEIGHTS = {
    "NS":       dict(pred=1.00, distill=0.00, obs=0.00, spec=0.00, acf=0.00, smooth=0.02, stab=0.01),
    "Emulator": dict(pred=0.55, distill=0.30, obs=0.30, spec=0.05, acf=0.05, smooth=0.02, stab=0.01),
    "Adaptive": dict(pred=0.55, distill=0.30, obs=0.30, spec=0.05, acf=0.05, smooth=0.02, stab=0.01),
}
STAB_RHO_FRACTION, ACF_MAX_LAG = 0.90, 8
LR, WEIGHT_DECAY, GRAD_CLIP = 8e-4, 1e-4, 1.0
LR_GRID = [8e-4, 3e-4]                             # validation-based LR selection
LR_SEARCH_MODELS = ["Transformer", "PatchTST", "TiDE", "TimesNet", "iTransformer"]
STAT_ALPHA = 0.05
TEACHER_EXPLODE_SIGMA = 4.0     # local (batch-level) gate threshold, in train-σ units
TEACHER_CLIP_SIGMA = 4.0        # winsorisation of teacher targets before distillation
RELIABLE_R, UNRELIABLE_R = 0.9, 0.5

BASELINES = ["Persistence", "MovingAverage", "MLP", "LSTM", "Transformer",
             "N-HiTS", "N-BEATS", "DLinear", "NLinear",
             "PatchTST", "TiDE", "TimesNet", "iTransformer"]
CENN_MODELS = (["CeNN_NS"] + [f"CeNN_Emu[{t}]" for t in TEACHERS]
                           + [f"CeNN_Ada[{t}]" for t in TEACHERS])
MODEL_ORDER = BASELINES + TEACHERS + CENN_MODELS               # 26 models
# Reduced sets for the omnibus analysis and the multi-horizon protocol (cost + CD power)
MH_MODELS = ["Persistence", "MLP", "DLinear", "NLinear", "N-BEATS", "TiDE",
             "QELM", "CeNN_NS", "CeNN_Emu[QELM]", "CeNN_Ada[QELM]"]
OMNI_MODELS = MH_MODELS

# --- Pre-registered confirmatory families (per-window paired tests, Holm within family)
PREREG = {
  "F1_negative_transfer": "CeNN_Ada[t] vs CeNN_Emu[t], per teacher t and dataset (one-sided: Ada <= Emu)",
  "F2_cost_of_distillation": "CeNN_Ada[t] vs CeNN_NS, per teacher t and dataset (two-sided)",
  "F3_competitiveness": "CeNN_Ada[best t] vs {DLinear, Persistence} per dataset (two-sided, descriptive)",
  "omnibus": "Friedman + Nemenyi over dataset x horizon blocks (N=18) on OMNI_MODELS (k=10)",
  "divergence_rule": "a run is 'diverged' if its test MASE > 3 x Persistence MASE on the same split",
}
json.dump({**CFG, "loss_weights": LOSS_WEIGHTS, "window": WINDOW, "main_horizon": MAIN_HORIZON,
           "multi_horizons": MULTI_HORIZONS, "teachers": TEACHERS, "models": MODEL_ORDER,
           "lr": LR, "lr_grid": LR_GRID, "lr_search_models": LR_SEARCH_MODELS,
           "teacher_clip_sigma": TEACHER_CLIP_SIGMA, "teacher_explode_sigma": TEACHER_EXPLODE_SIGMA,
           "reliable_r": RELIABLE_R, "unreliable_r": UNRELIABLE_R,
           "prereg": PREREG, "run_mode": RUN_MODE},
          open(f"{RESULTS_DIR}/config_dump.json", "w"), indent=2)
print("Mode:", RUN_MODE, "| seeds:", CFG["seeds"], "| epochs:", CFG["epochs"],
      "| teachers:", TEACHERS, "| models:", len(MODEL_ORDER), "| device:", DEVICE)

# ============================ v4 CONFIGURATION ================================
# Everything below is pre-registered: it is dumped to config_v4.json BEFORE any
# v4 training and must not be edited after the full run has started (R2, R8).

# --- R7: rolling-origin temporal validation ---
ORIGIN_TEST_FRAC  = 0.08     # length of each non-overlapping test block (fraction of series)
ORIGIN_CAL_FRAC   = 0.10     # gate-calibration block immediately preceding each test block
ORIGIN_SCHEME     = "expanding"   # training window grows with the origin index
MIN_TRAIN_POINTS  = WINDOW + MAIN_HORIZON + 500   # shorter series yield fewer origins

# --- R4: student-relative gate thresholds (three-action policy) ---
TAU_REJECT, TAU_ACCEPT = 0.5, 0.9   # reject below, accept above, attenuate between

# --- R14: safety endpoints ---
EPSILON_HARM   = 0.0     # regret tolerance (MASE units) defining "negative transfer"
NONINF_MARGIN  = 0.05    # relative-regret non-inferiority margin (5% of standalone MASE)

# --- R6: primary students / teachers of the v4 benchmark ---
V4_PROFILES = {
    "smoke": dict(origins=2, seeds=[42],           students=["DLinear"]),
    "fast":  dict(origins=3, seeds=[42, 123],      students=["DLinear", "N-HiTS"]),
    "full":  dict(origins=5, seeds=[42, 123, 2024], students=["DLinear", "PatchTST", "N-HiTS"]),
}
V4 = V4_PROFILES[RUN_MODE]
V4_TEACHERS = ["QELM", "QELM-NISQ", "ClassicalELM"]   # ideal / noisy / classical (Section 15-bis substrate)

# Trained gate policies; ValSelect / HardRejectPersist / Oracle are derived post-hoc (R5)
V4_POLICIES = ["NaiveKD", "SoftSR", "HardRejectSR", "LegacyPersist"]

PREREG_V4 = {
 "H1_harm_control":  "SoftSR regret vs NS is stochastically smaller than LegacyPersist regret and NaiveKD regret; "
                     "unit = dataset x origin (seeds kept as a level); one-sided Wilcoxon over origin blocks, Holm within family",
 "H2_incremental":   "SoftSR is non-inferior to HardRejectSR on mean relative regret (margin 5%) and improves "
                     "benefit retention on gate-accepted cells",
 "H3_noninferiority": "on gate-accepted cells (r_stu >= tau_accept), SoftSR relative regret vs best(NS, NaiveKD) "
                      "has bootstrap upper 95% CI below the 5% margin",
 "H4_generality":    "H1 direction holds for every primary student; per-student effects reported with CIs",
 "sensitivity":      "per-origin Diebold-Mariano (HAC lag = h-1) and moving-block bootstrap (block = WINDOW + h)",
 "unit":             "dataset x origin (x seed as level); overlapping windows are repeated measures, never the inference unit",
 "divergence":       "run diverged if test MASE > 3 x persistence MASE on the same origin",
}
json.dump({"n_origins": V4["origins"], "test_frac": ORIGIN_TEST_FRAC, "cal_frac": ORIGIN_CAL_FRAC,
           "scheme": ORIGIN_SCHEME, "tau_reject": TAU_REJECT, "tau_accept": TAU_ACCEPT,
           "epsilon_harm": EPSILON_HARM, "noninf_margin": NONINF_MARGIN,
           "students": V4["students"], "teachers": V4_TEACHERS, "policies": V4_POLICIES,
           "seeds": V4["seeds"], "prereg": PREREG_V4, "run_mode": RUN_MODE,
           "frozen_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())},
          open(f"{RESULTS_DIR}/config_v4.json", "w"), indent=2)
print("v4 config frozen | origins:", V4["origins"], "| students:", V4["students"],
      "| teachers:", V4_TEACHERS, "| policies:", V4_POLICIES, "| seeds:", V4["seeds"])


Mode: full | seeds: [42, 123, 2024, 7, 99] | epochs: 60 | teachers: ['QELM', 'VQC', 'QRC', 'QKRR'] | models: 26 | device: cuda
v4 config frozen | origins: 5 | students: ['DLinear', 'PatchTST', 'N-HiTS'] | teachers: ['QELM', 'QELM-NISQ', 'ClassicalELM'] | policies: ['NaiveKD', 'SoftSR', 'HardRejectSR', 'LegacyPersist'] | seeds: [42, 123, 2024]


## 3. Data: six real datasets, downloaded inside the notebook
Sources are tried in order: (i) Kaggle input datasets if attached, (ii) canonical public URLs.
Raw-file SHA-256 checksums are recorded for provenance. AAPL closing prices are converted to
log-returns (standard for financial series). All splits are chronological (70/15/15) with
**train-only normalisation** (no leakage).

In [3]:
DATASET_SOURCES = {
    "ETTh1": dict(target=["OT"], transform="identity", date=["date", "Date"],
        kaggle=["/kaggle/input/ettdataset/ETTh1.csv", "/kaggle/input/ett-small/ETTh1.csv"],
        url="https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv"),
    "ETTm2": dict(target=["OT"], transform="identity", date=["date", "Date"],
        kaggle=["/kaggle/input/ettdataset/ETTm2.csv", "/kaggle/input/ett-small/ETTm2.csv"],
        url="https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm2.csv"),
    "Energy": dict(target=["PJME_MW"], transform="identity", date=["Datetime", "datetime", "Date"],
        kaggle=["/kaggle/input/hourly-energy-consumption/PJME_hourly.csv"],
        url="https://raw.githubusercontent.com/panambY/Hourly_Energy_Consumption/master/data/PJME_hourly.csv"),
    "ExchangeRate": dict(target=["OT"], transform="identity", noheader=True,
        kaggle=["/kaggle/input/exchange-rate/exchange_rate.txt"],
        url="https://raw.githubusercontent.com/laiguokun/multivariate-time-series-data/master/exchange_rate/exchange_rate.txt.gz"),
    "Jena": dict(target=["T (degC)"], transform="identity", date=["Date Time", "datetime", "date"],
        kaggle=["/kaggle/input/jena-climate/jena_climate_2009_2016.csv"],
        url="https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"),
    "AAPL": dict(target=["Close", "Adj Close"], transform="log_return", date=["Date", "date"],
        kaggle=["/kaggle/input/aapl-stock-data/AAPL.csv", "/kaggle/input/aapl-stock-price/AAPL.csv"],
        url=["https://stooq.com/q/d/l/?s=aapl.us&i=d",
             "https://raw.githubusercontent.com/matplotlib/sample_data/master/aapl.csv"]),
}

def _dl(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    raw = urllib.request.urlopen(req, timeout=90).read()
    if url.endswith(".gz"): return gzip.decompress(raw), raw
    if url.endswith(".zip"):
        z = zipfile.ZipFile(io.BytesIO(raw))
        return z.read([n for n in z.namelist() if n.endswith(".csv")][0]), raw
    return raw, raw

def _series_from_df(df, spec):
    if spec.get("noheader"): return df.iloc[:, -1].to_numpy(dtype=np.float64)
    for dc in spec.get("date", []):
        if dc in df.columns:
            try: df = df.sort_values(dc)
            except Exception: pass
            break
    col = next((c for c in spec["target"] if c in df.columns), df.columns[-1])
    s = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float64)
    if spec["transform"] == "log_return":
        s = s[np.isfinite(s)]; s = s[s > 0]; s = np.diff(np.log(s))
    return s

def load_real(name):
    spec = DATASET_SOURCES[name]
    for p in spec.get("kaggle", []):
        if os.path.exists(p):
            raw = open(p, "rb").read()
            df = pd.read_csv(p, header=None) if spec.get("noheader") else pd.read_csv(p)
            return _series_from_df(df, spec), f"kaggle:{p}", hashlib.sha256(raw).hexdigest()
    urls = spec["url"] if isinstance(spec["url"], list) else [spec["url"]]
    last = "no url"
    for u in urls:
        try:
            content, raw = _dl(u)
            df = (pd.read_csv(io.BytesIO(content), header=None) if spec.get("noheader")
                  else pd.read_csv(io.BytesIO(content)))
            s = _series_from_df(df, spec)
            if len(s) > WINDOW + max(MULTI_HORIZONS) + 200:
                return s, u, hashlib.sha256(raw).hexdigest()
            last = "series too short"
        except Exception as e:
            last = f"{type(e).__name__}"
    return None, f"MISSING ({last})", None

RAW_SERIES, SOURCE_LOG = {}, {}
for nm in DATASET_SOURCES:
    s, src, sha = load_real(nm)
    SOURCE_LOG[nm] = {"source": src, "sha256": sha, "n_points": (int(len(s)) if s is not None else 0)}
    if s is not None: RAW_SERIES[nm] = s
    print(f"{nm:<13} {('OK n='+str(len(s))) if s is not None else 'MISSING':<14} {src}")
assert RAW_SERIES, "No dataset could be loaded: enable Internet in Kaggle notebook settings."
AVAILABLE = list(RAW_SERIES.keys())
json.dump(SOURCE_LOG, open(f"{RESULTS_DIR}/dataset_sources.json", "w"), indent=2)
print("Coverage:", AVAILABLE)

# Dataset overview figure (paper appendix)
fig, axes = plt.subplots(2, 3, figsize=(FIG_W2, 3.0))
for ax, nm in zip(axes.ravel(), AVAILABLE):
    v = RAW_SERIES[nm][-CFG["max_points"]:]
    ax.plot(v, lw=0.4, color="#333333"); ax.set_title(nm); ax.set_xticks([])
for ax in axes.ravel()[len(AVAILABLE):]: ax.axis("off")
fig.suptitle("Datasets (last %d points, raw scale)" % CFG["max_points"], y=1.02)
fig.tight_layout(); savefig(fig, "fig_datasets_overview")

ETTh1         OK n=17420     https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv
ETTm2         OK n=69680     https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm2.csv
Energy        OK n=145366    https://raw.githubusercontent.com/panambY/Hourly_Energy_Consumption/master/data/PJME_hourly.csv
ExchangeRate  OK n=7588      https://raw.githubusercontent.com/laiguokun/multivariate-time-series-data/master/exchange_rate/exchange_rate.txt.gz
Jena          OK n=420551    https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip
AAPL          OK n=6080      https://raw.githubusercontent.com/matplotlib/sample_data/master/aapl.csv
Coverage: ['ETTh1', 'ETTm2', 'Energy', 'ExchangeRate', 'Jena', 'AAPL']
[fig] fig_datasets_overview.pdf / .png saved


## 4. Chronological windowing (train-only scaling, no leakage)

In [4]:
from dataclasses import dataclass
@dataclass
class DS:
    name: str; horizon: int
    train_x: np.ndarray; train_y: np.ndarray; val_x: np.ndarray; val_y: np.ndarray
    test_x: np.ndarray; test_y: np.ndarray; test_y_raw: np.ndarray
    train_raw: np.ndarray; val_y_raw: np.ndarray; scaler_mean: float; scaler_scale: float

def make_windows(vs, vr, window, horizon):
    xs, ys, yr, ep = [], [], [], []
    for s in range(0, len(vs) - window - horizon + 1):
        xs.append(vs[s:s+window]); ys.append(vs[s+window:s+window+horizon])
        yr.append(vr[s+window:s+window+horizon]); ep.append(s+window+horizon-1)
    return (np.asarray(xs, np.float32), np.asarray(ys, np.float32),
            np.asarray(yr, np.float32), np.asarray(ep))

def prepare(name, series, horizon, max_points):
    v = series[np.isfinite(series)]
    if max_points and len(v) > max_points: v = v[-max_points:]
    n = len(v); tc = int(n*SPLIT_RATIOS[0]); vc = int(n*(SPLIT_RATIOS[0]+SPLIT_RATIOS[1]))
    train_raw = v[:tc]; mean = float(np.mean(train_raw)); scale = float(np.std(train_raw)+1e-8)
    vscaled = (v-mean)/scale
    x, y, yraw, ep = make_windows(vscaled, v, WINDOW, horizon)
    trm, vam, tem = ep < tc, (ep >= tc) & (ep < vc), ep >= vc
    return DS(name, horizon, x[trm], y[trm], x[vam], y[vam], x[tem], y[tem], yraw[tem],
              train_raw, yraw[vam], mean, scale)

def inverse_scale(vs, d): return np.asarray(vs)*d.scaler_scale + d.scaler_mean
print("windowing ready | window =", WINDOW)

# ==================== v4: ROLLING-ORIGIN PREPARATION (R7) =====================
ORIGIN_BOUNDS = []   # exported to origin_boundaries.csv (required evidence, R7)

def prepare_origins(name, series, horizon, max_points, n_origins=None):
    # Builds up to n_origins DS objects whose TEST blocks partition the tail of the
    # series without overlap. Scheme: expanding training window; the calibration
    # block (stored in the DS val_* fields, hence transparently consumed by the
    # reliability functions) immediately precedes each test block, so gate
    # calibration never sees test data. Scaling is refit on each origin's training
    # segment only: no leakage across origins.
    n_origins = n_origins or V4["origins"]
    v = series[np.isfinite(series)]
    if max_points and len(v) > max_points: v = v[-max_points:]
    n = len(v)
    test_len = max(int(n * ORIGIN_TEST_FRAC), WINDOW + horizon + 50)
    cal_len  = max(int(n * ORIGIN_CAL_FRAC),  WINDOW + horizon + 50)
    origins = []
    for k in range(n_origins):
        te_end, te_start = n - k * test_len, n - (k + 1) * test_len
        cal_end, cal_start = te_start, te_start - cal_len
        tr_end = cal_start
        if tr_end < MIN_TRAIN_POINTS: break     # short series -> fewer origins (declared)
        train_raw = v[:tr_end]
        mean = float(np.mean(train_raw)); scale = float(np.std(train_raw) + 1e-8)
        x, y, yraw, ep = make_windows((v - mean) / scale, v, WINDOW, horizon)
        trm = ep < tr_end
        cam = (ep >= cal_start) & (ep < cal_end)
        tem = (ep >= te_start)  & (ep < te_end)
        if trm.sum() < 200 or cam.sum() < 20 or tem.sum() < 20: break
        origins.append(DS(f"{name}@o{k}", horizon, x[trm], y[trm], x[cam], y[cam],
                          x[tem], y[tem], yraw[tem], train_raw, yraw[cam], mean, scale))
        ORIGIN_BOUNDS.append(dict(dataset=name, origin=k, scheme=ORIGIN_SCHEME,
                                  train_end=int(tr_end), cal_start=int(cal_start),
                                  cal_end=int(cal_end), test_start=int(te_start),
                                  test_end=int(te_end), n_train_windows=int(trm.sum()),
                                  n_cal_windows=int(cam.sum()), n_test_windows=int(tem.sum())))
    origins.reverse()   # chronological order: o(K-1) ... o0 relabelled oldest-first
    for i, d_ in enumerate(origins):
        d_.name = f"{name}@o{i}"
    return origins

# Self-test: test blocks must be pairwise disjoint and posterior to their calibration block.
_probe = prepare_origins(AVAILABLE[0], RAW_SERIES[AVAILABLE[0]], MAIN_HORIZON, CFG["max_points"])
_bnds = [b for b in ORIGIN_BOUNDS if b["dataset"] == AVAILABLE[0]]
for a in range(len(_bnds)):
    assert _bnds[a]["cal_end"] <= _bnds[a]["test_start"]
    for b in range(a + 1, len(_bnds)):
        assert (_bnds[a]["test_end"] <= _bnds[b]["test_start"]
                or _bnds[b]["test_end"] <= _bnds[a]["test_start"])
print(f"rolling origins ready | {AVAILABLE[0]}: {len(_probe)} origins, disjoint test blocks verified")


windowing ready | window = 96
rolling origins ready | ETTh1: 5 origins, disjoint test blocks verified


## 5. Four QML teacher families on one exact statevector substrate
All teachers are **exact statevector simulations** (no shot noise, no hardware noise), batched in
PyTorch and GPU-capable, sharing one 6-qubit, 3-layer data re-uploading circuit substrate
(RY data rotations, RZ circuit parameters, CNOT ring entangler):

| Teacher | Family | Trained part | Observables exposed for distillation |
|---|---|---|---|
| **QELM** | quantum extreme learning machine | ridge readout only (circuit frozen random) | per-layer ⟨Z⟩, ⟨ZZ⟩, ⟨X⟩ (54-d) |
| **VQC**  | variational quantum circuit | circuit angles + linear head (Adam, end-to-end) | per-layer ⟨Z⟩, ⟨ZZ⟩, ⟨X⟩ (54-d) |
| **QRC**  | quantum reservoir computing | ridge readout on the observable trajectory | trajectory ⟨Z⟩, ⟨ZZ⟩, ⟨X⟩ (16 steps × 18-d) |
| **QKRR** | quantum fidelity-kernel ridge regression | kernel ridge on Nyström landmarks | final-state ⟨Z⟩, ⟨ZZ⟩, ⟨X⟩ (18-d) |

The QRC uses a pure-state surrogate of **amplitude damping** (per-step one-qubit reset) to obtain the
fading-memory property required of reservoirs. The unified API (`fit`, `predict`, `observables`)
makes the distillation pipeline strictly **teacher-agnostic** — a prerequisite for claim C3.

In [5]:
C64 = torch.complex64

class StatevectorSim:
    """Exact n-qubit statevector simulator with batched, per-sample angles."""
    def __init__(self, n_qubits, device="cpu"):
        self.n = n_qubits; self.dim = 2 ** n_qubits; self.device = device
        idx = torch.arange(self.dim, device=device)
        self.z_signs = torch.stack(
            [1.0 - 2.0 * ((idx >> (self.n - 1 - q)) & 1).float() for q in range(self.n)])
        self.x_flips = torch.stack([idx ^ (1 << (self.n - 1 - q)) for q in range(self.n)])
        self.cnot_perms = []
        for c in range(self.n):
            t = (c + 1) % self.n
            cbit = (idx >> (self.n - 1 - c)) & 1
            self.cnot_perms.append(torch.where(cbit.bool(), idx ^ (1 << (self.n - 1 - t)), idx))

    def init_state(self, batch):
        psi = torch.zeros(batch, self.dim, dtype=C64, device=self.device); psi[:, 0] = 1.0; return psi

    def _1q(self, psi, q, a, b, c, d):
        B = psi.shape[0]; L, R = 2 ** q, 2 ** (self.n - 1 - q)
        psi = psi.view(B, L, 2, R); s0, s1 = psi[:, :, 0, :], psi[:, :, 1, :]
        a, b, c, d = (t.view(B, 1, 1) for t in (a, b, c, d))
        return torch.stack((a * s0 + b * s1, c * s0 + d * s1), dim=2).reshape(B, self.dim)

    def ry(self, psi, q, theta):
        h = theta.to(psi.real.dtype) / 2
        cos, sin = torch.cos(h).to(C64), torch.sin(h).to(C64)
        return self._1q(psi, q, cos, -sin, sin, cos)

    def rz(self, psi, q, theta):
        h = theta.to(psi.real.dtype) / 2
        em = torch.exp(-1j * h.to(C64)); ep = torch.exp(1j * h.to(C64))
        zero = torch.zeros_like(em)
        return self._1q(psi, q, em, zero, zero, ep)

    def cnot_ring(self, psi):
        for perm in self.cnot_perms: psi = psi[:, perm]
        return psi

    def reset_qubit(self, psi, q):
        """Pure-state surrogate of full amplitude damping on qubit q: the |1>
        population is transferred to |0> (phase of the |0> branch preserved),
        so |1> -> |0> exactly and the norm is conserved."""
        B = psi.shape[0]; L, R = 2 ** q, 2 ** (self.n - 1 - q)
        psi = psi.view(B, L, 2, R); s0, s1 = psi[:, :, 0, :], psi[:, :, 1, :]
        mag = torch.sqrt(s0.real ** 2 + s0.imag ** 2 + s1.real ** 2 + s1.imag ** 2)
        phase = s0 / torch.clamp(torch.sqrt(s0.real ** 2 + s0.imag ** 2), min=1e-12).to(C64)
        new0 = mag.to(C64) * phase
        return torch.stack((new0, torch.zeros_like(new0)), dim=2).reshape(B, self.dim)

    def observables(self, psi):
        """[<Z_q>, <Z_q Z_{q+1}>, <X_q>] for all q -> (B, 3n), real-valued."""
        prob = psi.real ** 2 + psi.imag ** 2
        z = prob @ self.z_signs.T
        zz = torch.stack([prob @ (self.z_signs[q] * self.z_signs[(q + 1) % self.n])
                          for q in range(self.n)], dim=1)
        x = torch.stack([(psi.conj() * psi[:, self.x_flips[q]]).real.sum(1)
                         for q in range(self.n)], dim=1)
        return torch.cat([z, zz, x], dim=1)

def _layered_circuit(sim, psi, angles, var_rz):
    """Data re-uploading circuit; returns final state + per-layer observables."""
    obs = []
    for l in range(angles.shape[1]):
        for q in range(sim.n):
            psi = sim.ry(psi, q, angles[:, l, q])
            psi = sim.rz(psi, q, var_rz[l, q].expand(psi.shape[0]))
        psi = sim.cnot_ring(psi)
        obs.append(sim.observables(psi))
    return psi, obs

N_QUBITS, N_LAYERS = 6, 3
print(f"statevector substrate ready: {N_QUBITS} qubits (dim {2**N_QUBITS}), {N_LAYERS} layers, device={DEVICE}")

statevector substrate ready: 6 qubits (dim 64), 3 layers, device=cuda


In [6]:
class _EncoderMixin:
    """Shared bounded random-projection encoder: window W -> (L, n) data angles."""
    def _make_encoder(self, in_dim, rng):
        self.P = torch.tensor(rng.normal(0, 1/np.sqrt(in_dim), (in_dim, self.layers*self.nq)),
                              dtype=torch.float32, device=self.device)
    def _encode(self, x):
        xt = torch.as_tensor(np.asarray(x, np.float32), device=self.device)
        return (np.pi * torch.tanh(xt @ self.P)).view(-1, self.layers, self.nq)

class QELMTeacher(_EncoderMixin):
    """Untrained random circuit; ridge readout on per-layer observables."""
    name = "QELM"
    def __init__(self, seed=0, device="cpu", n_qubits=N_QUBITS, layers=N_LAYERS, ridge=1e-3, batch=4096):
        self.nq, self.layers, self.ridge = n_qubits, layers, ridge
        self.seed, self.device, self.batch = seed, device, batch
        self.sim = StatevectorSim(n_qubits, device)
    def _features(self, x):
        outs = []
        with torch.no_grad():
            for i in range(0, len(x), self.batch):
                ang = self._encode(x[i:i+self.batch])
                psi = self.sim.init_state(ang.shape[0])
                _, obs = _layered_circuit(self.sim, psi, ang, self.var_rz)
                outs.append(torch.cat(obs, dim=1).cpu().numpy())
        return np.concatenate(outs, 0).astype(np.float32)
    def fit(self, x, y, x_val=None, y_val=None):
        rng = np.random.default_rng(self.seed); self._make_encoder(x.shape[1], rng)
        self.var_rz = torch.tensor(rng.uniform(0, 2*np.pi, (self.layers, self.nq)),
                                   dtype=torch.float32, device=self.device)
        H = self._features(x)
        A = H.T @ H + self.ridge * np.eye(H.shape[1], dtype=np.float32)
        self.beta = np.linalg.solve(A, H.T @ y).astype(np.float32); return self
    def predict(self, x): return (self._features(x) @ self.beta).astype(np.float32)
    def observables(self, x): return self._features(x)

class VQCTeacher(_EncoderMixin):
    """Variational circuit + linear head, trained end-to-end with Adam."""
    name = "VQC"
    def __init__(self, seed=0, device="cpu", n_qubits=N_QUBITS, layers=N_LAYERS,
                 epochs=30, lr=5e-2, batch=1024, patience=8):
        self.nq, self.layers, self.seed, self.device = n_qubits, layers, seed, device
        self.epochs, self.lr, self.batch, self.patience = epochs, lr, batch, patience
        self.sim = StatevectorSim(n_qubits, device)
    def _forward(self, ang):
        psi = self.sim.init_state(ang.shape[0])
        _, obs = _layered_circuit(self.sim, psi, ang, self.var_rz)
        feats = torch.cat(obs, dim=1)
        return feats @ self.head_w + self.head_b, feats
    def fit(self, x, y, x_val=None, y_val=None):
        rng = np.random.default_rng(self.seed); torch.manual_seed(self.seed)
        self._make_encoder(x.shape[1], rng)
        h, d = y.shape[1], 3*self.nq*self.layers
        self.var_rz = torch.tensor(rng.uniform(0, 2*np.pi, (self.layers, self.nq)),
                                   dtype=torch.float32, device=self.device, requires_grad=True)
        self.head_w = torch.zeros(d, h, device=self.device, requires_grad=True)
        self.head_b = torch.zeros(h, device=self.device, requires_grad=True)
        opt = torch.optim.Adam([self.var_rz, self.head_w, self.head_b], lr=self.lr)
        yt = torch.as_tensor(np.asarray(y, np.float32), device=self.device)
        n = len(x); best, bestv, bad = None, float("inf"), 0
        has_val = x_val is not None and len(x_val) > 0
        for ep in range(self.epochs):
            perm = torch.randperm(n)
            for i in range(0, n, self.batch):
                idx = perm[i:i+self.batch].numpy()
                pred, _ = self._forward(self._encode(x[idx]))
                loss = torch.mean((pred - yt[idx]) ** 2)
                if not torch.isfinite(loss): continue
                opt.zero_grad(); loss.backward(); opt.step()
            if has_val:
                with torch.no_grad():
                    pv, _ = self._forward(self._encode(x_val))
                    vl = float(torch.mean((pv - torch.as_tensor(
                        np.asarray(y_val, np.float32), device=self.device)) ** 2))
                if np.isfinite(vl) and vl < bestv:
                    bestv, bad = vl, 0
                    best = (self.var_rz.detach().clone(), self.head_w.detach().clone(),
                            self.head_b.detach().clone())
                else:
                    bad += 1
                    if bad >= self.patience: break
        if best is not None:
            self.var_rz = best[0].requires_grad_(False); self.head_w, self.head_b = best[1], best[2]
        else:
            self.var_rz = self.var_rz.detach()
            self.head_w = self.head_w.detach(); self.head_b = self.head_b.detach()
        return self
    def _batched(self, x):
        outs, feats = [], []
        with torch.no_grad():
            for i in range(0, len(x), 4096):
                p, f = self._forward(self._encode(x[i:i+4096]))
                outs.append(p.cpu().numpy()); feats.append(f.cpu().numpy())
        return (np.concatenate(outs, 0).astype(np.float32),
                np.concatenate(feats, 0).astype(np.float32))
    def predict(self, x): return self._batched(x)[0]
    def observables(self, x): return self._batched(x)[1]

class QRCTeacher:
    """Quantum reservoir: per-step injection, fixed dynamics, one-qubit reset
    (dissipation surrogate) per step, ridge readout on the trajectory."""
    name = "QRC"
    def __init__(self, seed=0, device="cpu", n_qubits=N_QUBITS, n_steps=16,
                 ridge=1e-3, batch=2048, inject_scale=np.pi):
        self.nq, self.steps, self.ridge = n_qubits, n_steps, ridge
        self.seed, self.device, self.batch, self.scale = seed, device, batch, inject_scale
        self.sim = StatevectorSim(n_qubits, device)
    def _features(self, x):
        x = np.asarray(x, np.float32); W = x.shape[1]
        edges = np.linspace(0, W, self.steps + 1).astype(int)
        seg = np.tanh(np.stack([x[:, a:b].mean(1) for a, b in zip(edges[:-1], edges[1:])], 1))
        outs = []
        with torch.no_grad():
            for i in range(0, len(seg), self.batch):
                sb = torch.as_tensor(seg[i:i+self.batch], device=self.device)
                B = sb.shape[0]; psi = self.sim.init_state(B); traj = []
                for t in range(self.steps):
                    for q in range(self.nq):
                        psi = self.sim.ry(psi, q, self.scale * self.inj[q] * sb[:, t])
                        psi = self.sim.rz(psi, q, self.fix_rz[q].expand(B))
                    psi = self.sim.cnot_ring(psi)
                    traj.append(self.sim.observables(psi))
                    psi = self.sim.reset_qubit(psi, t % self.nq)
                outs.append(torch.cat(traj, dim=1).cpu().numpy())
        return np.concatenate(outs, 0).astype(np.float32)
    def fit(self, x, y, x_val=None, y_val=None):
        rng = np.random.default_rng(self.seed)
        self.inj = torch.tensor(rng.uniform(0.4, 1.0, self.nq), dtype=torch.float32, device=self.device)
        self.fix_rz = torch.tensor(rng.uniform(0, 2*np.pi, self.nq), dtype=torch.float32, device=self.device)
        H = self._features(x)
        A = H.T @ H + self.ridge * np.eye(H.shape[1], dtype=np.float32)
        self.beta = np.linalg.solve(A, H.T @ y).astype(np.float32); return self
    def predict(self, x): return (self._features(x) @ self.beta).astype(np.float32)
    def observables(self, x): return self._features(x)

class QKRRTeacher(_EncoderMixin):
    """Fidelity-kernel ridge regression on a Nystroem landmark subset."""
    name = "QKRR"
    def __init__(self, seed=0, device="cpu", n_qubits=N_QUBITS, layers=N_LAYERS,
                 ridge=1e-2, n_landmarks=512, batch=4096):
        self.nq, self.layers, self.ridge = n_qubits, layers, ridge
        self.M, self.seed, self.device, self.batch = n_landmarks, seed, device, batch
        self.sim = StatevectorSim(n_qubits, device)
    def _states(self, x):
        outs = []
        with torch.no_grad():
            for i in range(0, len(x), self.batch):
                ang = self._encode(x[i:i+self.batch])
                psi = self.sim.init_state(ang.shape[0])
                psi, _ = _layered_circuit(self.sim, psi, ang, self.var_rz)
                outs.append(psi)
        return torch.cat(outs, 0)
    def fit(self, x, y, x_val=None, y_val=None):
        rng = np.random.default_rng(self.seed); self._make_encoder(x.shape[1], rng)
        self.var_rz = torch.tensor(rng.uniform(0, 2*np.pi, (self.layers, self.nq)),
                                   dtype=torch.float32, device=self.device)
        idx = rng.choice(len(x), size=min(self.M, len(x)), replace=False)
        self.land_psi = self._states(np.asarray(x, np.float32)[idx])
        G = ((self.land_psi @ self.land_psi.conj().T).abs() ** 2).real.cpu().numpy().astype(np.float64)
        self.alpha = np.linalg.solve(G + self.ridge*np.eye(len(idx)),
                                     np.asarray(y, np.float64)[idx]).astype(np.float32)
        return self
    def predict(self, x):
        psi = self._states(x)
        K = ((psi @ self.land_psi.conj().T).abs() ** 2).real.cpu().numpy().astype(np.float32)
        return (K @ self.alpha).astype(np.float32)
    def observables(self, x):
        outs = []
        with torch.no_grad():
            for i in range(0, len(x), self.batch):
                ang = self._encode(x[i:i+self.batch])
                psi = self.sim.init_state(ang.shape[0])
                psi, _ = _layered_circuit(self.sim, psi, ang, self.var_rz)
                outs.append(self.sim.observables(psi).cpu().numpy())
        return np.concatenate(outs, 0).astype(np.float32)

def make_teacher(kind, seed):
    if kind == "QELM": return QELMTeacher(seed=seed, device=DEVICE)
    if kind == "VQC":  return VQCTeacher(seed=seed, device=DEVICE, epochs=CFG["vqc_epochs"])
    if kind == "QRC":  return QRCTeacher(seed=seed, device=DEVICE)
    if kind == "QKRR": return QKRRTeacher(seed=seed, device=DEVICE, n_landmarks=CFG["qkrr_landmarks"])
    raise ValueError(kind)
print("teachers ready:", TEACHERS)

teachers ready: ['QELM', 'VQC', 'QRC', 'QKRR']


### 5.1 Simulator unit tests (executed, not just claimed)

In [7]:
def _probe_device():
    """cuda.is_available() can be True while no kernel exists for the GPU arch
    (e.g. Kaggle P100/sm_60 with recent PyTorch wheels) -> probe with a real op."""
    if torch.cuda.is_available():
        try:
            _ = (torch.arange(8, device="cuda").float() @ torch.ones(8, device="cuda")).item()
            return "cuda"
        except Exception as e:
            print(f"[warn] CUDA visible but unusable ({type(e).__name__}): falling back to CPU.\n"
                  "       On Kaggle, pick 'GPU T4 x2' — recent PyTorch wheels dropped "
                  "Pascal (P100/sm_60) kernels.")
    return "cpu"
DEVICE = _probe_device()

## 6. Neuro-symbolic CeNN student
The student is a Cellular Neural Network (locally-coupled lattice dynamics, shared 3-tap circular
kernel, `tanh` saturation) with two heads: a **forecast head** and an **emulation head** whose
target is the teacher's vector of **quantum observables** — symbolically meaningful physical
quantities (Pauli expectations), not opaque activations. The symbolic priors are explicit loss
terms: a **spectral-radius stability bound** on the lattice states, a **temporal smoothness**
penalty, and **spectral / autocorrelation structure alignment**. The emulation head dimension
adapts automatically to the teacher family (54 for QELM/VQC, 288 for QRC, 18 for QKRR).

In [8]:
class CeNNCore(nn.Module):
    def __init__(self, window, n_cells, steps=4):
        super().__init__(); self.project = nn.Linear(window, n_cells)
        self.kernel = nn.Parameter(torch.tensor([0.15, 0.70, 0.15]).view(1, 1, 3))
        self.bias = nn.Parameter(torch.zeros(n_cells)); self.steps = steps
    def forward(self, x, return_states=False):
        s = torch.tanh(self.project(x)); states = [s]
        for _ in range(self.steps):
            z = F.pad(s.unsqueeze(1), (1, 1), mode="circular")
            s = torch.tanh(0.65*s + F.conv1d(z, self.kernel).squeeze(1) + self.bias); states.append(s)
        return (s, states) if return_states else s

class CeNNForecaster(nn.Module):
    def __init__(self, window, horizon, n_cells, emul_dim, hidden):
        super().__init__(); self.core = CeNNCore(window, n_cells)
        self.forecast_head = nn.Sequential(nn.Linear(n_cells, hidden), nn.ReLU(), nn.Linear(hidden, horizon))
        self.emulation_head = nn.Sequential(nn.Linear(n_cells, hidden), nn.ReLU(), nn.Linear(hidden, emul_dim))
    def forward(self, x, return_aux=False):
        s, states = self.core(x, return_states=True)
        y = self.forecast_head(s); e = self.emulation_head(s)
        return (y, e, states) if return_aux else y
print("CeNN student ready")

CeNN student ready


## 7. Composite loss and the reliability gate (the contribution)
The gate has two levels. **Global** `r_global ∈ (0,1]` compares the teacher's validation MASE to the
naive-persistence MASE on the same validation windows: `r = 1/(1 + max(0, MASE_T/MASE_naive − 1))`,
so a teacher that cannot beat persistence is progressively silenced. **Local** (per batch) the gate
measures the fraction of teacher targets inside a ±4σ envelope, silencing exploding targets.
Additionally, teacher targets are **winsorised at ±4σ** before entering the distillation terms, so a
single exploding target cannot dominate a batch gradient. The gate multiplies only the four
alignment terms (distill / obs / spec / acf); the symbolic priors (smooth / stab) are never gated.

In [9]:
def l_pred(p, y): return torch.mean((p-y)**2)
def l_distill(p, tp): return torch.mean((p-tp)**2)
def l_obs(e, to): return torch.mean((e-to)**2)
def l_spec(p, tp): return torch.mean((torch.abs(torch.fft.rfft(p, dim=-1))
                                      - torch.abs(torch.fft.rfft(tp, dim=-1)))**2)
def _acf(seq, L):
    h = seq.shape[1]; L = min(L, h-1); c = seq - seq.mean(-1, keepdim=True)
    var = torch.sum(c**2, -1) + 1e-8
    return torch.stack([torch.sum(c[:, t:]*c[:, :-t], -1)/var for t in range(1, L+1)], -1)
def l_acf(p, tp, L): return torch.mean((_acf(p, L) - _acf(tp, L))**2)
def l_smooth(st): return sum(torch.mean((st[k]-st[k-1])**2) for k in range(1, len(st)))/max(1, len(st)-1)
def l_stab(st, rho): return sum(torch.mean(torch.clamp(torch.linalg.norm(s, dim=-1)-rho, min=0.0)**2)
                                for s in st)/len(st)

def composite_loss(pred, emul, states, y, tp, to, w, rho, acf_lag, gate=1.0):
    tot = w["pred"]*l_pred(pred, y)
    if w["distill"] > 0: tot = tot + gate*w["distill"]*l_distill(pred, tp)
    if w["obs"] > 0:     tot = tot + gate*w["obs"]*l_obs(emul, to)
    if w["spec"] > 0:    tot = tot + gate*w["spec"]*l_spec(pred, tp)
    if w["acf"] > 0:     tot = tot + gate*w["acf"]*l_acf(pred, tp, acf_lag)
    if w["smooth"] > 0:  tot = tot + w["smooth"]*l_smooth(states)
    if w["stab"] > 0:    tot = tot + w["stab"]*l_stab(states, rho)
    return tot

def teacher_reliability(teacher, d):
    """r_global in (0,1]: 1 if teacher <= naive persistence on validation."""
    yv = inverse_scale(d.val_y, d).ravel()
    qv = inverse_scale(teacher.predict(d.val_x), d).ravel()
    naive = inverse_scale(np.repeat(d.val_x[:, -1:], d.horizon, 1), d).ravel()
    denom = np.mean(np.abs(np.asarray(d.train_raw[1:]) - np.asarray(d.train_raw[:-1]))) + 1e-8
    mase_t = np.mean(np.abs(qv-yv))/denom; mase_n = np.mean(np.abs(naive-yv))/denom
    r = 1.0/(1.0 + max(0.0, mase_t/max(mase_n, 1e-8) - 1.0))
    return float(np.clip(r, 0.05, 1.0)), float(mase_t), float(mase_n)

def local_gate(teacher_batch):
    with torch.no_grad():
        return float((teacher_batch.abs() <= TEACHER_EXPLODE_SIGMA).float().mean())

def winsorize(a, s=TEACHER_CLIP_SIGMA):
    return np.clip(np.nan_to_num(np.asarray(a, np.float32), nan=0.0, posinf=s, neginf=-s), -s, s)
print("losses + reliability gate ready")

# ============ v4: STUDENT-RELATIVE GATE AND THREE-ACTION POLICY (R4) ==========
# The v3 persistence-relative score is kept verbatim above as the LegacyPersist
# baseline (its 0.05 floor is part of the criticised design and must be preserved
# for the comparison to be faithful). The primary gate below differs in three
# audited respects: (i) the reference is the standalone student, not persistence;
# (ii) the floor is removed, so rejection is exact (g = 0); (iii) the score feeds
# a three-action policy rather than a direct multiplicative weight.

def calibration_mase(pred_raw, d):
    # MASE of raw-scale predictions on the calibration block of origin d.
    yv = np.asarray(d.val_y_raw, float).ravel()
    pv = np.asarray(pred_raw, float).ravel()
    m = np.isfinite(yv) & np.isfinite(pv)
    return float(np.mean(np.abs(pv[m] - yv[m])) / mase_denom(d.train_raw))

def student_relative_reliability(teacher, d, ns_cal_mase):
    # r_stu in [0, 1]; equals 1 iff the teacher matches or beats the standalone
    # student on the calibration block. ns_cal_mase comes from the NS student
    # already trained on this (origin, seed): no additional training pass.
    tv = inverse_scale(teacher.predict(d.val_x), d)
    mase_t = calibration_mase(tv, d)
    r = 1.0 / (1.0 + max(0.0, mase_t / max(ns_cal_mase, 1e-8) - 1.0))
    return float(np.clip(r, 0.0, 1.0)), mase_t

def gate_policy(r_stu, tau_reject=None, tau_accept=None):
    # Three-action policy (R4): reject (exact g = 0) below tau_reject, accept
    # (g = 1) above tau_accept, linear attenuation in between.
    tr_ = TAU_REJECT if tau_reject is None else tau_reject
    ta_ = TAU_ACCEPT if tau_accept is None else tau_accept
    if r_stu < tr_:  return 0.0, "reject"
    if r_stu >= ta_: return 1.0, "accept"
    return float((r_stu - tr_) / (ta_ - tr_)), "attenuate"

def hard_reject_sr(r_stu):
    # Parameter-free hard rejection: admit the teacher iff it is at least as good
    # as the standalone student on calibration (r_stu = 1 by construction).
    return (1.0 if r_stu >= 1.0 - 1e-9 else 0.0)

# Unit tests of the policy logic (reviewer-facing).
assert gate_policy(0.10) == (0.0, "reject")
assert gate_policy(0.95) == (1.0, "accept")
_g, _a = gate_policy(0.70); assert _a == "attenuate" and abs(_g - 0.5) < 1e-9
assert hard_reject_sr(1.0) == 1.0 and hard_reject_sr(0.999) == 0.0
print("student-relative gate + three-action policy ready - unit tests PASSED")


losses + reliability gate ready
student-relative gate + three-action policy ready - unit tests PASSED


## 8. Robust training loops (NaN guards, divergence logging, gate telemetry)

In [10]:
def make_loader(tensors, bs, shuffle):
    ds = torch.utils.data.TensorDataset(*[torch.as_tensor(t) for t in tensors])
    g = torch.Generator(); g.manual_seed(0)
    return torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=shuffle, generator=g)

GATE_LOG = []   # dataset, teacher, seed, epoch, gate (telemetry for fig_gate_dynamics)

def train_cenn(model, d, seed, teacher, kind, teacher_name="", lr=None):
    """kind in {'NS','Emulator','Adaptive'}. Teacher targets are winsorised at +/-4 sigma."""
    set_seed(seed); model = model.to(DEVICE)
    w = LOSS_WEIGHTS[kind]; rho = STAB_RHO_FRACTION*np.sqrt(CFG["n_cells"])
    if kind == "NS":
        tr = np.zeros_like(d.train_y); ob = np.zeros((len(d.train_x), 8), np.float32)
        tv = np.zeros_like(d.val_y);  ov = np.zeros((len(d.val_x), 8), np.float32)
        r_global = 1.0
    else:
        tr = winsorize(teacher.predict(d.train_x)); ob = winsorize(teacher.observables(d.train_x))
        tv = winsorize(teacher.predict(d.val_x));   ov = winsorize(teacher.observables(d.val_x))
        r_global, _, _ = teacher_reliability(teacher, d) if kind == "Adaptive" else (1.0, None, None)
    loader = make_loader([d.train_x, d.train_y, tr, ob], CFG["batch_size"], True)
    opt = torch.optim.AdamW(model.parameters(), lr=(lr or LR), weight_decay=WEIGHT_DECAY)
    best, bestv, bad = None, float("inf"), 0
    for ep in range(CFG["epochs"]):
        model.train(); ep_gates = []
        for xb, yb, tb, obb in loader:
            xb, yb, tb, obb = (t.to(DEVICE) for t in (xb, yb, tb, obb))
            gate = r_global*local_gate(tb) if kind == "Adaptive" else 1.0
            ep_gates.append(gate)
            opt.zero_grad(set_to_none=True)
            p, e, st = model(xb, return_aux=True)
            loss = composite_loss(p, e, st, yb, tb, obb, w, rho, ACF_MAX_LAG, gate=gate)
            if not torch.isfinite(loss): continue
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP); opt.step()
        if kind == "Adaptive":
            GATE_LOG.append(dict(dataset=d.name, teacher=teacher_name, seed=seed,
                                 epoch=ep, gate=float(np.mean(ep_gates)),
                                 r_global=float(r_global),
                                 distill_weight=float(w["distill"]*r_global)))
        model.eval()
        with torch.no_grad():
            p, e, st = model(torch.as_tensor(d.val_x).to(DEVICE), return_aux=True)
            vl = composite_loss(p, e, st, torch.as_tensor(d.val_y).to(DEVICE),
                    torch.as_tensor(tv).to(DEVICE), torch.as_tensor(ov).to(DEVICE),
                    w, rho, ACF_MAX_LAG, gate=r_global if kind == "Adaptive" else 1.0)
            vl = float(vl) if torch.isfinite(vl) else float("inf")
        if vl < bestv:
            bestv, bad = vl, 0
            best = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= CFG["patience"]: break
    if best is not None: model.load_state_dict(best)
    return model, r_global

def train_baseline(model, d, seed, lr=None):
    set_seed(seed); model = model.to(DEVICE)
    loader = make_loader([d.train_x, d.train_y], CFG["batch_size"], True)
    opt = torch.optim.AdamW(model.parameters(), lr=(lr or LR), weight_decay=WEIGHT_DECAY)
    mse = nn.MSELoss(); best, bestv, bad = None, float("inf"), 0
    for _ in range(CFG["epochs"]):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad(set_to_none=True)
            loss = mse(model(xb), yb)
            if not torch.isfinite(loss): continue
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP); opt.step()
        model.eval()
        with torch.no_grad():
            vl = float(mse(model(torch.as_tensor(d.val_x).to(DEVICE)),
                           torch.as_tensor(d.val_y).to(DEVICE)))
        if np.isfinite(vl) and vl < bestv:
            bestv, bad = vl, 0
            best = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= CFG["patience"]: break
    if best is not None: model.load_state_dict(best)
    return model, bestv

@torch.no_grad()
def predict_torch(model, x, aux=False):
    model.eval(); xb = torch.as_tensor(x).to(DEVICE)
    if aux:
        p, e, _ = model(xb, return_aux=True); return p.cpu().numpy(), e.cpu().numpy()
    return model(xb).cpu().numpy()
print("training loops ready")

training loops ready


## 9. Metrics, effect sizes, statistical utilities (with unit tests)

In [11]:
def mase_denom(tr, m=1):
    return np.nan if len(tr) <= m else float(np.mean(np.abs(tr[m:]-tr[:-m]))+1e-8)

def compute_metrics(yt_raw, yp_raw, tr):
    yt = np.asarray(yt_raw, float).ravel(); yp = np.asarray(yp_raw, float).ravel()
    m = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[m], yp[m]
    if len(yt) == 0:
        return dict(MAE=np.nan, RMSE=np.nan, MSE=np.nan, MASE=np.nan, Bias=np.nan, WAPE=np.nan)
    err = yp-yt; ae = np.abs(err); md_ = mase_denom(tr)
    return dict(MAE=float(np.mean(ae)), RMSE=float(np.sqrt(np.mean(err**2))), MSE=float(np.mean(err**2)),
                MASE=float(np.mean(ae)/md_), Bias=float(np.mean(err)),
                WAPE=float(np.sum(ae)/(np.sum(np.abs(yt))+1e-8)*100))

def per_window_mae(yt_raw, yp_raw):
    """Mean absolute error per test window -> (N,) : the unit of confirmatory tests."""
    yt = np.asarray(yt_raw, float); yp = np.asarray(yp_raw, float)
    e = np.abs(yp - yt); e[~np.isfinite(e)] = np.nan
    return np.nanmean(e, axis=1)

def safe_corr(a, b, method="pearson"):
    a = np.asarray(a).ravel(); b = np.asarray(b).ravel()
    m = np.isfinite(a) & np.isfinite(b); a, b = a[m], b[m]
    if len(a) < 3 or np.std(a) < 1e-12 or np.std(b) < 1e-12: return np.nan
    return float(stats.spearmanr(a, b).correlation if method == "spearman" else stats.pearsonr(a, b)[0])

def spectral_distance(a, b):
    a = np.asarray(a).ravel(); b = np.asarray(b).ravel(); n = min(len(a), len(b), 4096)
    if n < 8: return np.nan
    A = np.abs(np.fft.rfft(a[:n]-a[:n].mean())); B = np.abs(np.fft.rfft(b[:n]-b[:n].mean()))
    A /= (np.linalg.norm(A)+1e-12); B /= (np.linalg.norm(B)+1e-12)
    return float(np.linalg.norm(A-B))

def emulation_metrics(teacher, model, level):
    t = np.asarray(teacher).ravel(); m = np.asarray(model).ravel()
    return dict(level=level, emul_Pearson_r=safe_corr(t, m), emul_Spearman_r=safe_corr(t, m, "spearman"),
                emul_MAE_to_teacher=float(np.mean(np.abs(m-t))),
                emul_RMSE_to_teacher=float(np.sqrt(np.mean((m-t)**2))),
                emul_spectral_distance=spectral_distance(t, m))

def cliffs_delta(a, b, max_n=1500, seed=0):
    """Vectorised Cliff's delta with sub-sampling for O(n^2) tractability."""
    a = np.asarray(a, float); b = np.asarray(b, float)
    a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
    if len(a) == 0 or len(b) == 0: return np.nan, "undefined"
    rng = np.random.default_rng(seed)
    if len(a) > max_n: a = rng.choice(a, max_n, replace=False)
    if len(b) > max_n: b = rng.choice(b, max_n, replace=False)
    diff = a[:, None] - b[None, :]
    d = float((np.sum(diff > 0) - np.sum(diff < 0)) / (len(a)*len(b))); ad = abs(d)
    return d, ("negligible" if ad < 0.147 else "small" if ad < 0.33
               else "medium" if ad < 0.474 else "large")

def holm(pv):
    p = np.asarray(pv, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.0
    for r, i in enumerate(order):
        run = max(run, (m-r)*p[i]); adj[i] = min(run, 1.0)
    return adj

def bootstrap_ci(v, iters=None, alpha=0.05, seed=0):
    v = np.asarray(v, float); v = v[np.isfinite(v)]; iters = iters or CFG["bootstrap"]
    if len(v) < 2: return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    m = [rng.choice(v, len(v), True).mean() for _ in range(iters)]
    return (float(np.percentile(m, 100*alpha/2)), float(np.percentile(m, 100*(1-alpha/2))))

# --- Unit tests of the statistical machinery (reviewer-facing) ---------------
_adj = holm([0.01, 0.04, 0.03])
assert np.allclose(_adj, [0.03, 0.06, 0.06]), _adj          # textbook Holm example
_d, _lab = cliffs_delta(np.zeros(50), np.ones(50))
assert _d == -1.0 and _lab == "large"
_d0, _ = cliffs_delta(np.arange(100), np.arange(100))
assert abs(_d0) < 1e-9
assert abs(per_window_mae(np.array([[0., 0.], [1., 1.]]),
                          np.array([[1., 3.], [1., 1.]]))[0] - 2.0) < 1e-9
print("statistical utilities ready - unit tests PASSED")

statistical utilities ready - unit tests PASSED


## 10. Baseline model zoo (13 classical & deep forecasters)

In [12]:
class Persistence:
    def fit(self, x, y): return self
    def predict(self, x, h): return np.repeat(x[:, -1:], h, 1).astype(np.float32)
class MovingAverage:
    def __init__(self, k=12): self.k = k
    def fit(self, x, y): return self
    def predict(self, x, h): return np.repeat(x[:, -self.k:].mean(1, keepdims=True), h, 1).astype(np.float32)

class MLP(nn.Module):
    def __init__(self, w, h, hidden=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(w, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, h))
    def forward(self, x): return self.net(x)
class LSTMReg(nn.Module):
    def __init__(self, w, h, hidden=64):
        super().__init__(); self.lstm = nn.LSTM(1, hidden, batch_first=True); self.head = nn.Linear(hidden, h)
    def forward(self, x): o, _ = self.lstm(x.unsqueeze(-1)); return self.head(o[:, -1])
class TransformerReg(nn.Module):
    def __init__(self, w, h, d=64, nh=4, L=2):
        super().__init__(); self.embed = nn.Linear(1, d); self.pos = nn.Parameter(torch.randn(1, w, d)*0.02)
        self.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(d, nh, 128, batch_first=True, dropout=0.1), L)
        self.head = nn.Linear(d, h)
    def forward(self, x): return self.head(self.enc(self.embed(x.unsqueeze(-1))+self.pos).mean(1))
class DLinear(nn.Module):
    def __init__(self, w, h, k=25):
        super().__init__(); self.k = k; self.trend = nn.Linear(w, h); self.seasonal = nn.Linear(w, h)
    def _ma(self, x):
        p = self.k//2
        return F.avg_pool1d(F.pad(x.unsqueeze(1), (p, p), mode="replicate"), self.k, 1).squeeze(1)
    def forward(self, x): t = self._ma(x); return self.trend(t)+self.seasonal(x-t)
class NLinear(nn.Module):
    def __init__(self, w, h):
        super().__init__(); self.lin = nn.Linear(w, h)
    def forward(self, x): last = x[:, -1:]; return self.lin(x-last)+last
class NBeats(nn.Module):
    def __init__(self, w, h, hidden=128, blocks=3):
        super().__init__(); self.w, self.h = w, h
        self.blocks = nn.ModuleList([nn.Sequential(nn.Linear(w, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, w+h)) for _ in range(blocks)])
    def forward(self, x):
        res = x; fc = torch.zeros(x.shape[0], self.h, device=x.device)
        for b in self.blocks:
            o = b(res); res = res-o[:, :self.w]; fc = fc+o[:, self.w:]
        return fc
class NHiTS(nn.Module):
    def __init__(self, w, h, hidden=128, pools=(4, 2, 1)):
        super().__init__(); self.w, self.h, self.pools = w, h, pools
        self.blocks = nn.ModuleList([nn.ModuleDict({
            "mlp": nn.Sequential(nn.Linear(max(1, w//p), hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU()),
            "back": nn.Linear(hidden, w), "fore": nn.Linear(hidden, max(1, h//p))}) for p in pools])
    def forward(self, x):
        res = x; fc = torch.zeros(x.shape[0], self.h, device=x.device)
        for p, blk in zip(self.pools, self.blocks):
            pooled = F.avg_pool1d(res.unsqueeze(1), p).squeeze(1) if p > 1 else res
            hh = blk["mlp"](pooled); res = res-blk["back"](hh)
            f = F.interpolate(blk["fore"](hh).unsqueeze(1), size=self.h,
                              mode="linear", align_corners=False).squeeze(1)
            fc = fc+f
        return fc
class PatchTST(nn.Module):
    def __init__(self, w, h, patch=16, stride=8, d=64, nh=4, L=2):
        super().__init__(); self.patch, self.stride = patch, stride; self.np = (w-patch)//stride + 1
        self.embed = nn.Linear(patch, d); self.pos = nn.Parameter(torch.randn(1, self.np, d)*0.02)
        self.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(d, nh, 128, batch_first=True, dropout=0.1), L)
        self.head = nn.Linear(self.np*d, h)
    def forward(self, x):
        p = x.unfold(1, self.patch, self.stride)
        z = self.enc(self.embed(p)+self.pos)
        return self.head(z.reshape(z.shape[0], -1))
class TiDE(nn.Module):
    def __init__(self, w, h, hidden=128):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(w, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.dec = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, h))
        self.skip = nn.Linear(w, h)
    def forward(self, x): return self.dec(self.enc(x)) + self.skip(x)
class TimesNet(nn.Module):
    def __init__(self, w, h, d=32, k=6):
        super().__init__(); self.w, self.h = w, h; self.embed = nn.Linear(1, d)
        self.conv = nn.Sequential(nn.Conv2d(d, d, 3, padding=1), nn.GELU(), nn.Conv2d(d, d, 3, padding=1))
        self.head = nn.Linear(w*d, h)
    def forward(self, x):
        B = x.shape[0]; z = self.embed(x.unsqueeze(-1))
        xf = torch.fft.rfft(x, dim=1); amp = torch.abs(xf); amp[:, 0] = 0
        period = max(2, int((x.shape[1] // (amp.mean(0).argmax().item()+1))))
        period = min(period, x.shape[1]); rows = x.shape[1] // period
        if rows < 1: rows, period = 1, x.shape[1]
        pad = rows*period
        zt = z[:, :pad].reshape(B, rows, period, -1).permute(0, 3, 1, 2)
        zt = self.conv(zt).permute(0, 2, 3, 1).reshape(B, pad, -1)
        if pad < self.w: zt = F.pad(zt, (0, 0, 0, self.w-pad))
        return self.head(zt.reshape(B, -1))
class ITransformer(nn.Module):
    def __init__(self, w, h, seg=12, d=64, nh=4, L=2):
        super().__init__(); self.seg = seg; self.ns = w//seg
        self.embed = nn.Linear(seg, d)
        self.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(d, nh, 128, batch_first=True, dropout=0.1), L)
        self.head = nn.Linear(self.ns*d, h)
    def forward(self, x):
        B = x.shape[0]; xs = x[:, :self.ns*self.seg].reshape(B, self.ns, self.seg)
        return self.head(self.enc(self.embed(xs)).reshape(B, -1))

TORCH_BASELINES = {"MLP": MLP, "LSTM": LSTMReg, "Transformer": TransformerReg,
    "N-HiTS": NHiTS, "N-BEATS": NBeats, "DLinear": DLinear, "NLinear": NLinear,
    "PatchTST": PatchTST, "TiDE": TiDE, "TimesNet": TimesNet, "iTransformer": ITransformer}

def build_model(name, horizon):
    cls = TORCH_BASELINES[name]
    if name in ("MLP", "LSTM", "N-HiTS", "N-BEATS"):
        return cls(WINDOW, horizon, hidden=CFG["hidden_dim"])
    return cls(WINDOW, horizon)
print("baseline zoo ready:", list(TORCH_BASELINES))

baseline zoo ready: ['MLP', 'LSTM', 'Transformer', 'N-HiTS', 'N-BEATS', 'DLinear', 'NLinear', 'PatchTST', 'TiDE', 'TimesNet', 'iTransformer']


> **LEGACY STUDY (v3, single 70/15/15 split).** Retained verbatim for the mechanistic CeNN section and the v3-comparison appendix; it carries no confirmatory claim in v4. The primary benchmark is Part III-bis.

## 11. Main benchmark (h = 24): 13 baselines + 4 QML teachers + 9 CeNN variants
- **Validation-based learning-rate selection** for the attention/SOTA baselines
  (`Transformer, PatchTST, TiDE, TimesNet, iTransformer`) over `LR_GRID`, chosen once on the
  first seed per dataset and reused — addressing the "under-tuned baselines" concern explicitly.
- **Per-window absolute errors are stored** for every (dataset, model, seed): these are the units
  of the confirmatory statistics of §13.
- **All artefacts are saved incrementally** after each dataset (a killed session keeps its results).

In [13]:
PW_ERRORS = {}   # (dataset, horizon, model, seed) -> per-window MAE vector (raw scale)
LR_CHOICE = {}   # (dataset, model) -> selected lr

def select_lr(name, d, seed):
    key = (d.name, name)
    if key in LR_CHOICE: return LR_CHOICE[key]
    if name not in LR_SEARCH_MODELS: LR_CHOICE[key] = LR; return LR
    best_lr, best_v = LR, float("inf")
    saved_epochs = CFG["epochs"]; CFG["epochs"] = max(6, saved_epochs//3)
    for lr in LR_GRID:
        _, v = train_baseline(build_model(name, d.horizon), d, seed, lr=lr)
        if np.isfinite(v) and v < best_v: best_v, best_lr = v, lr
    CFG["epochs"] = saved_epochs
    LR_CHOICE[key] = best_lr
    print(f"    [lr-select] {name} on {d.name}: {best_lr}")
    return best_lr

PRED_EXAMPLES = {}                                   # (dataset, model) -> [n_windows, horizon]
EXAMPLE_MODELS = ["Persistence", "DLinear", "CeNN_NS", "CeNN_Emu[QELM]", "CeNN_Ada[QELM]"]
EXAMPLE_N_WINDOWS = 3

def run_main(datasets, horizon):
    rows, emu, rel = [], [], []
    for name in datasets:
        d = prepare(name, RAW_SERIES[name], horizon, CFG["max_points"])
        print(f"\n=== {name} (h={horizon}) train={len(d.train_x)} val={len(d.val_x)} "
              f"test={len(d.test_x)} | elapsed {elapsed()} ===")
        for seed in CFG["seeds"]:
            set_seed(seed)
            teachers = {}
            for t in TEACHERS:
                tt0 = time.time()
                teachers[t] = make_teacher(t, seed).fit(d.train_x, d.train_y, d.val_x, d.val_y)
                rg, mt, mn = teacher_reliability(teachers[t], d)
                rel.append(dict(dataset=name, horizon=horizon, teacher=t, seed=seed,
                                r_global=rg, teacher_val_mase=mt, naive_val_mase=mn,
                                fit_seconds=round(time.time()-tt0, 1)))
            def record(m, pr, emul_head=None, teacher_name=None):
                met = compute_metrics(d.test_y_raw, pr, d.train_raw)
                rows.append(dict(dataset=name, horizon=horizon, model=m, seed=seed, **met))
                PW_ERRORS[(name, horizon, m, seed)] = per_window_mae(d.test_y_raw, pr).astype(np.float32)
                if seed == CFG["seeds"][0] and m in EXAMPLE_MODELS:      # for the qualitative figure
                    PRED_EXAMPLES[(name, m)] = np.asarray(pr)[:EXAMPLE_N_WINDOWS].astype(np.float32)
                    PRED_EXAMPLES[(name, "__truth__")] = np.asarray(d.test_y_raw)[:EXAMPLE_N_WINDOWS].astype(np.float32)
                if teacher_name is not None:
                    tea_raw = inverse_scale(winsorize(teachers[teacher_name].predict(d.test_x)), d)
                    emu.append(dict(dataset=name, horizon=horizon, model=m, teacher=teacher_name,
                                    seed=seed, **emulation_metrics(tea_raw, pr, "prediction")))
                    if emul_head is not None:
                        tea_obs = winsorize(teachers[teacher_name].observables(d.test_x))
                        emu.append(dict(dataset=name, horizon=horizon, model=m, teacher=teacher_name,
                                        seed=seed, **emulation_metrics(tea_obs, emul_head, "observable")))
            def fail(m, e):
                print(f"  [!] {m} failed ({type(e).__name__}: {e}) -> MASE=NaN")
                rows.append(dict(dataset=name, horizon=horizon, model=m, seed=seed,
                                 MAE=np.nan, RMSE=np.nan, MSE=np.nan, MASE=np.nan,
                                 Bias=np.nan, WAPE=np.nan))
            # -- naive + deep baselines
            for m in BASELINES:
                try:
                    if m == "Persistence": ps = Persistence().predict(d.test_x, horizon)
                    elif m == "MovingAverage": ps = MovingAverage(12).predict(d.test_x, horizon)
                    else:
                        lr = select_lr(m, d, seed) if seed == CFG["seeds"][0] else LR_CHOICE.get((name, m), LR)
                        model, _ = train_baseline(build_model(m, horizon), d, seed, lr=lr)
                        ps = predict_torch(model, d.test_x)
                    record(m, inverse_scale(ps, d))
                except Exception as e: fail(m, e)
            # -- teachers as forecasters
            for t in TEACHERS:
                try: record(t, inverse_scale(teachers[t].predict(d.test_x), d))
                except Exception as e: fail(t, e)
            # -- CeNN student: NS once, Emulator/Adaptive per teacher
            try:
                mdl = CeNNForecaster(WINDOW, horizon, CFG["n_cells"], 8, CFG["hidden_dim"])
                mdl, _ = train_cenn(mdl, d, seed, None, "NS")
                record("CeNN_NS", inverse_scale(predict_torch(mdl, d.test_x), d))
            except Exception as e: fail("CeNN_NS", e)
            for t in TEACHERS:
                obs_dim = teachers[t].observables(d.train_x[:2]).shape[1]
                for kind, tag in (("Emulator", f"CeNN_Emu[{t}]"), ("Adaptive", f"CeNN_Ada[{t}]")):
                    try:
                        mdl = CeNNForecaster(WINDOW, horizon, CFG["n_cells"], obs_dim, CFG["hidden_dim"])
                        mdl, rg = train_cenn(mdl, d, seed, teachers[t], kind, teacher_name=t)
                        ps, eh = predict_torch(mdl, d.test_x, aux=True)
                        record(tag, inverse_scale(ps, d), emul_head=eh, teacher_name=t)
                    except Exception as e: fail(tag, e)
            print(f"  seed {seed} done | elapsed {elapsed()}")
        # incremental persistence after each dataset
        pd.DataFrame(rows).to_csv(f"{RESULTS_DIR}/benchmark_full.csv", index=False)
        pd.DataFrame(emu).to_csv(f"{RESULTS_DIR}/emulation_fidelity_full.csv", index=False)
        pd.DataFrame(rel).to_csv(f"{RESULTS_DIR}/teacher_reliability.csv", index=False)
        np.savez_compressed(f"{RESULTS_DIR}/per_window_errors.npz",
                            **{"|".join(map(str, k)): v for k, v in PW_ERRORS.items()})
        pd.DataFrame(GATE_LOG).to_csv(f"{RESULTS_DIR}/gate_dynamics.csv", index=False)
        np.savez_compressed(f"{RESULTS_DIR}/prediction_examples.npz",
                            **{"|".join(k): v for k, v in PRED_EXAMPLES.items()})
    return pd.DataFrame(rows), pd.DataFrame(emu), pd.DataFrame(rel)

t0 = time.time()
results_df, emulation_df, reliability_df = run_main(AVAILABLE, MAIN_HORIZON)
print(f"\nMain benchmark done in {(time.time()-t0)/60:.1f} min | rows={len(results_df)}")


=== ETTh1 (h=24) train=12075 val=2613 test=2613 | elapsed 0.1 min ===
    [lr-select] Transformer on ETTh1: 0.0003
    [lr-select] PatchTST on ETTh1: 0.0008
    [lr-select] TiDE on ETTh1: 0.0008
    [lr-select] TimesNet on ETTh1: 0.0003
    [lr-select] iTransformer on ETTh1: 0.0008
  seed 42 done | elapsed 6.2 min
  seed 123 done | elapsed 11.0 min
  seed 2024 done | elapsed 15.5 min
  seed 7 done | elapsed 20.9 min
  seed 99 done | elapsed 27.0 min

=== ETTm2 (h=24) train=13881 val=3000 test=3000 | elapsed 27.0 min ===
    [lr-select] Transformer on ETTm2: 0.0008
    [lr-select] PatchTST on ETTm2: 0.0003
    [lr-select] TiDE on ETTm2: 0.0003
    [lr-select] TimesNet on ETTm2: 0.0008
    [lr-select] iTransformer on ETTm2: 0.0008
  seed 42 done | elapsed 36.8 min
  seed 123 done | elapsed 44.1 min
  seed 2024 done | elapsed 51.4 min
  seed 7 done | elapsed 59.8 min
  seed 99 done | elapsed 67.9 min

=== Energy (h=24) train=13881 val=3000 test=3000 | elapsed 67.9 min ===
    [lr-select]

## 12. Multi-horizon protocol (h = 48, 96) on the reduced model set

In [14]:
def run_multi_horizon():
    rows = []
    for horizon in [h for h in MULTI_HORIZONS if h != MAIN_HORIZON]:
        for name in AVAILABLE:
            d = prepare(name, RAW_SERIES[name], horizon, CFG["max_points"])
            print(f"=== MH {name} h={horizon} | elapsed {elapsed()} ===")
            for seed in CFG["mh_seeds"]:
                set_seed(seed)
                qt = make_teacher("QELM", seed).fit(d.train_x, d.train_y)
                def rec(m, pr):
                    met = compute_metrics(d.test_y_raw, pr, d.train_raw)
                    rows.append(dict(dataset=name, horizon=horizon, model=m, seed=seed, **met))
                    PW_ERRORS[(name, horizon, m, seed)] = per_window_mae(d.test_y_raw, pr).astype(np.float32)
                for m in MH_MODELS:
                    try:
                        if m == "Persistence": pr = inverse_scale(Persistence().predict(d.test_x, horizon), d)
                        elif m == "QELM": pr = inverse_scale(qt.predict(d.test_x), d)
                        elif m == "CeNN_NS":
                            mdl = CeNNForecaster(WINDOW, horizon, CFG["n_cells"], 8, CFG["hidden_dim"])
                            mdl, _ = train_cenn(mdl, d, seed, None, "NS")
                            pr = inverse_scale(predict_torch(mdl, d.test_x), d)
                        elif m.startswith("CeNN_"):
                            kind = "Emulator" if "Emu" in m else "Adaptive"
                            od = qt.observables(d.train_x[:2]).shape[1]
                            mdl = CeNNForecaster(WINDOW, horizon, CFG["n_cells"], od, CFG["hidden_dim"])
                            mdl, _ = train_cenn(mdl, d, seed, qt, kind, teacher_name="QELM")
                            pr = inverse_scale(predict_torch(mdl, d.test_x), d)
                        else:
                            mdl, _ = train_baseline(build_model(m, horizon), d, seed,
                                                    lr=LR_CHOICE.get((name, m), LR))
                            pr = inverse_scale(predict_torch(mdl, d.test_x), d)
                        rec(m, pr)
                    except Exception as e:
                        print(f"  [!] {m} h={horizon} failed: {type(e).__name__}")
                        rows.append(dict(dataset=name, horizon=horizon, model=m, seed=seed,
                                         MAE=np.nan, RMSE=np.nan, MSE=np.nan, MASE=np.nan,
                                         Bias=np.nan, WAPE=np.nan))
    return pd.DataFrame(rows)

if CFG["multi_horizon"]:
    mh_extra = run_multi_horizon()
    base_h = results_df[results_df.model.isin(MH_MODELS)
                        & results_df.seed.isin(CFG["mh_seeds"])].copy()
    multih_df = pd.concat([base_h, mh_extra], ignore_index=True)
else:
    multih_df = results_df[results_df.model.isin(MH_MODELS)].copy()
multih_df.to_csv(f"{RESULTS_DIR}/benchmark_multihorizon.csv", index=False)
np.savez_compressed(f"{RESULTS_DIR}/per_window_errors.npz",
                    **{"|".join(map(str, k)): v for k, v in PW_ERRORS.items()})
print("multi-horizon rows:", len(multih_df))

=== MH ETTh1 h=48 | elapsed 164.6 min ===
=== MH ETTm2 h=48 | elapsed 167.6 min ===
=== MH Energy h=48 | elapsed 173.2 min ===
=== MH ExchangeRate h=48 | elapsed 178.9 min ===
=== MH Jena h=48 | elapsed 180.2 min ===
=== MH AAPL h=48 | elapsed 186.2 min ===
=== MH ETTh1 h=96 | elapsed 187.2 min ===
=== MH ETTm2 h=96 | elapsed 189.7 min ===
=== MH Energy h=96 | elapsed 195.4 min ===
=== MH ExchangeRate h=96 | elapsed 201.0 min ===
=== MH Jena h=96 | elapsed 202.5 min ===
=== MH AAPL h=96 | elapsed 206.9 min ===
multi-horizon rows: 540


## 13. Adaptive gating study — the central result, now per teacher family
For every teacher `t` and dataset we compare `CeNN_NS`, `CeNN_Emu[t]`, `CeNN_Ada[t]` alongside the
measured validation reliability `r_global`. Regimes: **reliable** (r ≥ 0.9), **intermediate**,
**unreliable** (r < 0.5).

In [15]:
gs_rows = []
mase_by = results_df[results_df.horizon == MAIN_HORIZON].groupby(["dataset", "model"])["MASE"].mean()
r_by = reliability_df.groupby(["dataset", "teacher"])["r_global"].mean()
for t in TEACHERS:
    for ds_ in AVAILABLE:
        try:
            ada = mase_by.get((ds_, f"CeNN_Ada[{t}]"), np.nan)
            emu_ = mase_by.get((ds_, f"CeNN_Emu[{t}]"), np.nan)
            ns  = mase_by.get((ds_, "CeNN_NS"), np.nan)
            rg  = r_by.get((ds_, t), np.nan)
            regime = ("reliable teacher" if rg >= RELIABLE_R else
                      "unreliable teacher" if rg < UNRELIABLE_R else "intermediate")
            gs_rows.append(dict(teacher=t, dataset=ds_, r_global=rg, regime=regime,
                                CeNN_NS=ns, CeNN_Emulator=emu_, CeNN_Adaptive=ada,
                                gain_vs_Emulator=emu_-ada, gain_vs_NS=ns-ada))
        except Exception as e:
            print("gating row failed", t, ds_, e)
gating_df = pd.DataFrame(gs_rows)
gating_df.to_csv(f"{RESULTS_DIR}/adaptive_gating_study.csv", index=False)
print(gating_df.round(3).to_string(index=False))

teacher      dataset  r_global             regime  CeNN_NS  CeNN_Emulator  CeNN_Adaptive  gain_vs_Emulator  gain_vs_NS
   QELM        ETTh1     0.103 unreliable teacher    2.239          4.012          2.263             1.750      -0.024
   QELM        ETTm2     0.290 unreliable teacher    8.981         45.748         32.190            13.558     -23.209
   QELM       Energy     0.751       intermediate    1.805          3.032          2.891             0.140      -1.087
   QELM ExchangeRate     0.085 unreliable teacher   17.440         63.443         20.795            42.649      -3.354
   QELM         Jena     0.436 unreliable teacher    8.436         17.455         14.544             2.912      -6.108
   QELM         AAPL     1.000   reliable teacher    0.601          0.600          0.600             0.000       0.001
    VQC        ETTh1     0.129 unreliable teacher    2.239          3.521          2.317             1.204      -0.078
    VQC        ETTm2     0.470 unreliable teache

> **DESCRIPTIVE ONLY in v4 (R8).** Per-window Wilcoxon on overlapping windows is anti-conservative under serial dependence; confirmatory inference lives in the dependence-aware cell of Part III-bis.

## 14. Statistics
**Confirmatory** (powered): per-window paired Wilcoxon signed-rank on seed-averaged absolute errors,
Holm-corrected inside each pre-registered family, with Cliff's delta effect sizes.
**Descriptive**: 26-model mean ranks; Friedman + Nemenyi over dataset × horizon blocks on the reduced
set; bootstrap CIs; divergence report with a sensitivity re-analysis excluding diverged runs.

In [16]:
def pw_series(ds_, h_, model, seeds):
    vs = [PW_ERRORS.get((ds_, h_, model, s)) for s in seeds]
    vs = [v for v in vs if v is not None]
    if not vs: return None
    L = min(len(v) for v in vs)
    return np.nanmean(np.stack([v[:L] for v in vs]), axis=0)

def paired_test(ds_, a_name, b_name, alternative="two-sided"):
    a = pw_series(ds_, MAIN_HORIZON, a_name, CFG["seeds"])
    b = pw_series(ds_, MAIN_HORIZON, b_name, CFG["seeds"])
    if a is None or b is None: return None
    L = min(len(a), len(b)); a, b = a[:L], b[:L]
    m = np.isfinite(a) & np.isfinite(b); a, b = a[m], b[m]
    if len(a) < 20: return None
    try: p = float(stats.wilcoxon(a, b, alternative=alternative, zero_method="wilcox").pvalue)
    except ValueError: p = 1.0
    dlt, eff = cliffs_delta(a, b)
    return dict(n_windows=int(len(a)), p_value=p, cliffs_delta=dlt, effect=eff,
                mean_diff=float(np.mean(a-b)), median_diff=float(np.median(a-b)))

stat_rows = []
# F1: Ada vs Emu (one-sided: Ada has smaller error), per teacher x dataset
for t in TEACHERS:
    for ds_ in AVAILABLE:
        r = paired_test(ds_, f"CeNN_Ada[{t}]", f"CeNN_Emu[{t}]", alternative="less")
        if r: stat_rows.append(dict(family="F1_negative_transfer", teacher=t, dataset=ds_,
                                    comparison=f"Ada[{t}] vs Emu[{t}]", **r))
# F2: Ada vs NS, two-sided
for t in TEACHERS:
    for ds_ in AVAILABLE:
        r = paired_test(ds_, f"CeNN_Ada[{t}]", "CeNN_NS", alternative="two-sided")
        if r: stat_rows.append(dict(family="F2_cost_of_distillation", teacher=t, dataset=ds_,
                                    comparison=f"Ada[{t}] vs NS", **r))
# F3: Ada[best teacher overall] vs DLinear / Persistence
best_t = gating_df.groupby("teacher")["CeNN_Adaptive"].mean().idxmin()
for ref in ("DLinear", "Persistence"):
    for ds_ in AVAILABLE:
        r = paired_test(ds_, f"CeNN_Ada[{best_t}]", ref, alternative="two-sided")
        if r: stat_rows.append(dict(family="F3_competitiveness", teacher=best_t, dataset=ds_,
                                    comparison=f"Ada[{best_t}] vs {ref}", **r))
stat_df = pd.DataFrame(stat_rows)
for fam in stat_df.family.unique():   # Holm within each pre-registered family
    m = stat_df.family == fam
    stat_df.loc[m, "p_holm"] = holm(stat_df.loc[m, "p_value"].values)
stat_df["significant"] = stat_df["p_holm"] < STAT_ALPHA
stat_df.to_csv(f"{RESULTS_DIR}/statistical_comparison_full.csv", index=False)
print(stat_df.groupby("family")["significant"].agg(["sum", "count"]))
print("\nbest teacher (mean Adaptive MASE):", best_t)

                         sum  count
family                             
F1_negative_transfer      20     24
F2_cost_of_distillation   23     24
F3_competitiveness        12     12

best teacher (mean Adaptive MASE): QRC


In [17]:
# --- Descriptive ranks (26 models), omnibus on blocks, bootstrap CIs, divergence
main_df = results_df[results_df.horizon == MAIN_HORIZON]
piv_all = main_df.pivot_table(index="dataset", columns="model", values="MASE", aggfunc="mean")
mean_ranks_all = piv_all.rank(axis=1).mean(0).sort_values()
mean_ranks_all.rename("mean_rank").to_csv(f"{RESULTS_DIR}/mean_ranks.csv")
print("Descriptive mean ranks (top 12):\n", mean_ranks_all.head(12).round(2))

# Omnibus over dataset x horizon blocks (k reduced => informative CD)
piv_b = multih_df.pivot_table(index=["dataset", "horizon"], columns="model",
                              values="MASE", aggfunc="mean")
piv_b = piv_b[[m for m in OMNI_MODELS if m in piv_b.columns]].dropna()
fried_stat, fried_p = stats.friedmanchisquare(*[piv_b[m].values for m in piv_b.columns])
ranks_b = piv_b.rank(axis=1); avg_ranks_b = ranks_b.mean(0)
k, Nb = piv_b.shape[1], piv_b.shape[0]
qT = {2:1.960,3:2.343,4:2.569,5:2.728,6:2.850,7:2.949,8:3.031,9:3.102,10:3.164,
      11:3.219,12:3.268,13:3.313,14:3.354,15:3.391,16:3.426,17:3.458}
CD = qT.get(k, 3.164)*np.sqrt(k*(k+1)/(6*Nb))
json.dump(dict(friedman_chi2=float(fried_stat), friedman_p=float(fried_p),
               nemenyi_CD=float(CD), k=int(k), N=int(Nb),
               avg_ranks={m: float(v) for m, v in avg_ranks_b.items()}),
          open(f"{RESULTS_DIR}/omnibus_tests.json", "w"), indent=2)
print(f"\nOmnibus: Friedman chi2={fried_stat:.2f} p={fried_p:.3g} | CD(k={k}, N={Nb})={CD:.2f}")

# Bootstrap CIs of dataset-mean MASE per model
ci_rows = []
for m in MODEL_ORDER:
    v = main_df[main_df.model == m].groupby("dataset")["MASE"].mean().values
    lo, hi = bootstrap_ci(v)
    ci_rows.append(dict(model=m, MASE_mean=float(np.nanmean(v)), ci_low=lo, ci_high=hi))
pd.DataFrame(ci_rows).to_csv(f"{RESULTS_DIR}/bootstrap_ci.csv", index=False)

# Divergence report (pre-registered rule) + sensitivity ranks
pers = main_df[main_df.model == "Persistence"].set_index(["dataset", "seed"])["MASE"]
div_rows = []
for _, r in main_df.iterrows():
    thr = pers.get((r.dataset, r.seed), np.nan)
    div_rows.append(bool(np.isfinite(r.MASE) and np.isfinite(thr) and r.MASE > 3*thr) or not np.isfinite(r.MASE))
main_df = main_df.assign(diverged=div_rows)
div_tab = main_df.groupby("model")["diverged"].agg(["sum", "count"])
div_tab.to_csv(f"{RESULTS_DIR}/divergence_report.csv")
clean = main_df[~main_df.diverged]
piv_c = clean.pivot_table(index="dataset", columns="model", values="MASE", aggfunc="mean")
pd.concat([mean_ranks_all.rename("rank_all"),
           piv_c.rank(axis=1).mean(0).rename("rank_no_diverged")], axis=1)\
  .to_csv(f"{RESULTS_DIR}/rank_sensitivity.csv")
print("\nDiverged runs per model (top):\n", div_tab.sort_values("sum", ascending=False).head(8))

# summary mean +/- std table
smm = main_df.pivot_table(index="model", columns="dataset", values="MASE", aggfunc="mean")
sms = main_df.pivot_table(index="model", columns="dataset", values="MASE", aggfunc="std")
summary = smm.round(3).astype(str) + " +/- " + sms.round(3).astype(str)
summary.reindex(MODEL_ORDER).to_csv(f"{RESULTS_DIR}/summary_mean_std.csv")
smd = main_df.pivot_table(index="model", columns="dataset", values="MASE", aggfunc="median")
smd.reindex(MODEL_ORDER).round(3).to_csv(f"{RESULTS_DIR}/summary_median.csv")
print("\nsummary tables saved")

Descriptive mean ranks (top 12):
 model
MLP              4.83
DLinear          4.83
NLinear          5.17
LSTM             6.33
TimesNet         6.33
N-HiTS           7.50
TiDE             7.50
N-BEATS          8.17
Transformer      9.17
iTransformer     9.83
PatchTST        10.00
CeNN_NS         10.83
dtype: float64

Omnibus: Friedman chi2=74.58 p=1.91e-12 | CD(k=10, N=18)=3.19

Diverged runs per model (top):
                 sum  count
model                     
QELM             11     30
CeNN_Emu[QELM]   10     30
VQC               9     30
QKRR              8     30
CeNN_Ada[QELM]    7     30
CeNN_Emu[VQC]     6     30
CeNN_Emu[QKRR]    5     30
CeNN_Ada[QKRR]    5     30

summary tables saved


## 15. Loss-term ablation (QELM-Adaptive, cumulative components)

In [18]:

# ====== Loss-term ablation (legacy CeNN) + NESTED CONFIGURATION SELECTION =====
# v4 addition (R13): the deployed configuration is selected per dataset by
# CALIBRATION MASE inside the split - never by test performance. The Complete
# configuration remains in the table as a finding: its instability (v3, Table V)
# is the design evidence motivating the simplified primary objective used by the
# strong students in Part III-bis.
BASE_W = dict(pred=1.0, distill=0.0, obs=0.0, spec=0.0, acf=0.0, smooth=0.0, stab=0.0)
ABL = {
    "F":            dict(BASE_W),
    "F+S":          dict(BASE_W, smooth=0.02),
    "F+S+St":       dict(BASE_W, smooth=0.02, stab=0.01),
    "F+S+St+D":     dict(BASE_W, smooth=0.02, stab=0.01, distill=0.30),
    "F+S+St+D+Obs": dict(BASE_W, smooth=0.02, stab=0.01, distill=0.30, obs=0.30),
    "Complete":     dict(LOSS_WEIGHTS["Adaptive"]),
}
abl_rows = []
for name in AVAILABLE:
    d = prepare(name, RAW_SERIES[name], MAIN_HORIZON, CFG["max_points"])
    for seed in CFG["abl_seeds"]:
        set_seed(seed)
        qt = make_teacher("QELM", seed).fit(d.train_x, d.train_y)
        od = qt.observables(d.train_x[:2]).shape[1]
        for cname, w in ABL.items():
            try:
                saved = LOSS_WEIGHTS["Adaptive"]; LOSS_WEIGHTS["Adaptive"] = w
                mdl = CeNNForecaster(WINDOW, MAIN_HORIZON, CFG["n_cells"], od, CFG["hidden_dim"])
                mdl, _ = train_cenn(mdl, d, seed, qt, "Adaptive", teacher_name="QELM")
                LOSS_WEIGHTS["Adaptive"] = saved
                pr = inverse_scale(predict_torch(mdl, d.test_x), d)
                pc = inverse_scale(predict_torch(mdl, d.val_x), d)
                mase = compute_metrics(d.test_y_raw, pr, d.train_raw)["MASE"]
                cal = calibration_mase(pc, d)
            except Exception as e:
                LOSS_WEIGHTS["Adaptive"] = saved; mase, cal = np.nan, np.nan
                print("ablation failed", name, cname, type(e).__name__)
            abl_rows.append(dict(dataset=name, config=cname, seed=seed,
                                 MASE=mase, cal_MASE=cal))
    print(f"ablation {name} done | elapsed {elapsed()}")
ablation_df = pd.DataFrame(abl_rows)
ablation_df.to_csv(f"{RESULTS_DIR}/ablation_losses.csv", index=False)

# Nested selection: per dataset, the config with the lowest mean calibration MASE
# is the pre-declared choice; its TEST MASE is then read once.
sel_rows = []
for name, g in ablation_df.groupby("dataset"):
    cal = g.groupby("config")["cal_MASE"].mean().dropna()
    if not len(cal): continue
    pick = cal.idxmin()
    sel_rows.append(dict(dataset=name, selected_config=pick,
                         selected_cal_MASE=float(cal[pick]),
                         selected_test_MASE=float(
                             g[g.config == pick]["MASE"].mean()),
                         complete_test_MASE=float(
                             g[g.config == "Complete"]["MASE"].mean())))
objective_selection = pd.DataFrame(sel_rows)
objective_selection.to_csv(f"{RESULTS_DIR}/objective_selection.csv", index=False)
print("\n=== nested objective selection (calibration-based, R13) ===")
print(objective_selection.round(3).to_string(index=False))
abl_sum = ablation_df.groupby(["dataset", "config"])["MASE"].agg(["mean", "std"]).reset_index()
abl_sum.to_csv(f"{RESULTS_DIR}/ablation_losses_summary.csv", index=False)
print(abl_sum.round(3).to_string(index=False))


ablation ETTh1 done | elapsed 211.8 min
ablation ETTm2 done | elapsed 217.0 min
ablation Energy done | elapsed 223.1 min
ablation ExchangeRate done | elapsed 225.4 min
ablation Jena done | elapsed 229.8 min
ablation AAPL done | elapsed 230.6 min

=== nested objective selection (calibration-based, R13) ===
     dataset selected_config  selected_cal_MASE  selected_test_MASE  complete_test_MASE
        AAPL        Complete              0.624               0.599               0.599
       ETTh1               F              2.157               2.292               2.190
       ETTm2             F+S              8.331               8.972              32.175
      Energy               F              1.531               1.802               2.934
ExchangeRate             F+S             35.325              17.269              22.353
        Jena             F+S              7.888               8.588              14.678
     dataset       config   mean   std
        AAPL     Complete  0.599 0.001

## Part III-bis — PRIMARY BENCHMARK (v4): rolling origins x strong students x gate policies

This section is the confirmatory core of the RINENG submission. It crosses (R5-R7):

- **Origins**: up to 5 non-overlapping test blocks per series, expanding training window,
  per-origin gate calibration on a block that precedes the test block (`prepare_origins`).
- **Students** (R6): `DLinear`, `PatchTST`, `N-HiTS` — three strong, structurally different
  architectures. The CeNN remains in the legacy mechanistic study only.
- **Teachers**: `QELM` (ideal statevector), `QELM-NISQ` (finite shots + depolarizing surrogate),
  `ClassicalELM` (no quantum ingredient) — the tier structure required by R11.
- **Trained policies** (R4/R5): `NS` (standalone), `NaiveKD` (g = 1), `SoftSR`
  (student-relative three-action gate), `HardRejectSR` (parameter-free student-relative
  rejection), `LegacyPersist` (the v3 persistence-relative gate with its 0.05 floor, kept as
  the criticised baseline). `ValSelect`, `HardRejectPersist` and `Oracle` are derived
  post-hoc from these runs without additional training.

Every run logs its test MASE, calibration MASE, divergence flag, gate value and action, and a
raw per-window error shard (`results/pw_v4/`) — the inference cells below consume only these
artifacts, and the benchmark is checkpointed at (dataset, origin, seed) granularity so a
Kaggle 12 h session can resume without recomputation.

In [19]:
# --- 15-bis (a): NISQ-noise QELM + classical unreliable teacher -------------

class NISQQELMTeacher(QELMTeacher):
    '''QELM whose observables pass through a NISQ surrogate channel:
    per-layer depolarizing damping + finite-shot sampling noise.

    Depolarizing: a global depolarizing channel with rate p per layer maps any
    traceless observable expectation o -> (1-p)^L o after L layers; we damp
    each layer's observable block accordingly.
    Shots: each ideal expectation o in [-1,1] is replaced by a sample mean of
    n_shots +/-1 outcomes; by the CLT its variance is (1-o^2)/n_shots.
    '''
    name = "QELM-NISQ"
    def __init__(self, seed=0, device="cpu", n_qubits=N_QUBITS, layers=N_LAYERS,
                 ridge=1e-3, batch=4096, shots=1024, depol=0.01):
        super().__init__(seed=seed, device=device, n_qubits=n_qubits,
                         layers=layers, ridge=ridge, batch=batch)
        self.shots, self.depol = shots, depol
        self._noise_rng = np.random.default_rng(seed + 10_000)

    def _features(self, x):
        O = super()._features(x)                       # (B, 3*nq*layers), ideal
        blk = 3 * self.nq
        for l in range(self.layers):                   # depolarizing damping
            O[:, l*blk:(l+1)*blk] *= (1.0 - self.depol) ** (l + 1)
        if self.shots:                                 # finite-shot noise
            var = np.clip(1.0 - O**2, 0.0, None) / self.shots
            O = np.clip(O + self._noise_rng.normal(0.0, np.sqrt(var)).astype(np.float32),
                        -1.0, 1.0)
        return O.astype(np.float32)

class ClassicalELMTeacher:
    '''Purely classical analog of the QELM: random tanh features + ridge
    readout with a deliberately near-singular regularizer, so that its
    reliability is dataset-dependent (like the QML teachers) without any
    quantum ingredient. Same fit/predict/observables API as the QML teachers.
    '''
    name = "ClassicalELM"
    def __init__(self, seed=0, width=3*N_QUBITS*N_LAYERS, ridge=1e-8):
        self.seed, self.width, self.ridge = seed, width, ridge
    def fit(self, x, y, x_val=None, y_val=None):
        rng = np.random.default_rng(self.seed + 777)
        W = x.shape[1]
        self.A = rng.normal(0.0, 2.0/np.sqrt(W), (W, self.width)).astype(np.float32)
        self.b = rng.uniform(-np.pi, np.pi, self.width).astype(np.float32)
        H = self._features(x)
        A_ = H.T @ H + self.ridge * np.eye(self.width, dtype=np.float32)
        self.beta = np.linalg.solve(A_, H.T @ y).astype(np.float32)
        return self
    def _features(self, x):
        return np.tanh(np.asarray(x, np.float32) @ self.A + self.b)
    def predict(self, x): return (self._features(x) @ self.beta).astype(np.float32)
    def observables(self, x): return self._features(x)

GEN_TEACHER_FACTORY = {
    "QELM":         lambda seed: make_teacher("QELM", seed),
    "QELM-NISQ":    lambda seed: NISQQELMTeacher(seed=seed, device=DEVICE,
                                                 shots=1024, depol=0.01),
    "ClassicalELM": lambda seed: ClassicalELMTeacher(seed=seed),
}
GEN_TEACHERS = list(GEN_TEACHER_FACTORY)
print("15-bis teachers ready:", GEN_TEACHERS)


15-bis teachers ready: ['QELM', 'QELM-NISQ', 'ClassicalELM']


In [20]:

# --- 15-bis (b) [v4]: gated-distillation trainer for ANY torch student -------
# Identical optimisation protocol to train_cenn restricted to the terms a generic
# forecaster possesses (R13, simplified objective): L = L_pred + g * lambda * L_KD.
# v4 additions: (i) an ns_model short-circuit implementing the EXACT g = 0 state
# required by R4 - a rejected teacher yields the standalone student by
# construction, not by an attenuated approximation; (ii) r_override carries any
# gate policy, so every comparator shares this single training path (R5: only
# the decision policy differs across arms).

GEN_DISTILL_W = LOSS_WEIGHTS["Adaptive"]["distill"]   # 0.30, unchanged from v3

def train_distilled_student(model, d, seed, teacher, kind, lr=None,
                            r_override=None, ns_model=None):
    # kind in {NS, Emulator, Adaptive}. Teacher targets winsorised at +/-4 sigma.
    # Returns (model, effective_gate).
    if kind == "Adaptive" and r_override is not None and r_override <= 0.0 \
            and ns_model is not None:
        return ns_model, 0.0          # exact rejection: reuse the standalone student
    set_seed(seed); model = model.to(DEVICE)
    if kind == "NS":
        tr = np.zeros_like(d.train_y); tv = np.zeros_like(d.val_y); r_global = 1.0
    else:
        tr = winsorize(teacher.predict(d.train_x))
        tv = winsorize(teacher.predict(d.val_x))
        r_global = (r_override if r_override is not None else
                    (teacher_reliability(teacher, d)[0] if kind == "Adaptive" else 1.0))
    loader = make_loader([d.train_x, d.train_y, tr], CFG["batch_size"], True)
    opt = torch.optim.AdamW(model.parameters(), lr=(lr or LR), weight_decay=WEIGHT_DECAY)
    best, bestv, bad = None, float("inf"), 0
    for ep in range(CFG["epochs"]):
        model.train()
        for xb, yb, tb in loader:
            xb, yb, tb = (t.to(DEVICE) for t in (xb, yb, tb))
            gate = (r_global * local_gate(tb) if kind == "Adaptive"
                    else (1.0 if kind == "Emulator" else 0.0))
            opt.zero_grad(set_to_none=True)
            p = model(xb)
            loss = l_pred(p, yb) + gate * GEN_DISTILL_W * l_distill(p, tb)
            if not torch.isfinite(loss): continue
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP); opt.step()
        model.eval()
        with torch.no_grad():
            p = model(torch.as_tensor(d.val_x).to(DEVICE))
            g = r_global if kind == "Adaptive" else (1.0 if kind == "Emulator" else 0.0)
            vl = float(l_pred(p, torch.as_tensor(d.val_y).to(DEVICE))
                       + g * GEN_DISTILL_W * l_distill(p, torch.as_tensor(tv).to(DEVICE)))
        if np.isfinite(vl) and vl < bestv:
            bestv, bad = vl, 0
            best = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= CFG["patience"]: break
    if best is not None: model.load_state_dict(best)
    return model, r_global

# Equivalence self-test (R4, gate A of the specification): with g = 0 the trainer
# must return the standalone student object itself.
_d0 = _probe[0]
set_seed(0); _ns = DLinear(WINDOW, MAIN_HORIZON)
_m, _g = train_distilled_student(DLinear(WINDOW, MAIN_HORIZON), _d0, 0, None,
                                 "Adaptive", r_override=0.0, ns_model=_ns)
assert _m is _ns and _g == 0.0
print("generic gated trainer ready | exact-rejection equivalence PASSED "
      f"(distill weight = {GEN_DISTILL_W})")


generic gated trainer ready | exact-rejection equivalence PASSED (distill weight = 0.3)


In [21]:

# ============== v4 PRIMARY BENCHMARK: origins x students x policies ===========
PW4_DIR = os.path.join(RESULTS_DIR, "pw_v4"); os.makedirs(PW4_DIR, exist_ok=True)
V4_RUNS_CSV = f"{RESULTS_DIR}/v4_runs.csv"
V4_REL_CSV  = f"{RESULTS_DIR}/v4_reliability.csv"

def _v4_load(csv_path):
    return pd.read_csv(csv_path) if os.path.exists(csv_path) else pd.DataFrame()

_prev_runs, _prev_rel = _v4_load(V4_RUNS_CSV), _v4_load(V4_REL_CSV)
DONE_BLOCKS = (set(map(tuple, _prev_runs[["dataset", "origin", "seed"]]
                       .drop_duplicates().values)) if len(_prev_runs) else set())
print(f"checkpoint: {len(DONE_BLOCKS)} (dataset, origin, seed) blocks already complete")

V4_LR = {}
def v4_select_lr(s_name, d, seed):
    # Fair-tuning parity (R10): the same LR_GRID search applied in the legacy
    # benchmark is applied here, once per (dataset, student), shared across
    # origins and policies so that only the gate policy varies between arms.
    key = (d.name.split("@")[0], s_name)
    if key in V4_LR: return V4_LR[key]
    if s_name not in LR_SEARCH_MODELS: V4_LR[key] = LR; return LR
    best_lr, best_v = LR, float("inf")
    saved = CFG["epochs"]; CFG["epochs"] = max(6, saved // 3)
    for lr_ in LR_GRID:
        _, v = train_baseline(build_model(s_name, d.horizon), d, seed, lr=lr_)
        if np.isfinite(v) and v < best_v: best_v, best_lr = v, lr_
    CFG["epochs"] = saved; V4_LR[key] = best_lr
    print(f"    [v4 lr-select] {s_name} on {key[0]}: {best_lr}")
    return best_lr

def _v4_shard_path(ds_base, origin, seed):
    return os.path.join(PW4_DIR, f"{ds_base}__o{origin}__s{seed}.npz")

t0_v4 = time.time(); n_trained = n_reused = 0
for name in AVAILABLE:
    origins = prepare_origins(name, RAW_SERIES[name], MAIN_HORIZON, CFG["max_points"])
    print(f"\n=== [v4] {name}: {len(origins)} origins | elapsed {elapsed()} ===")
    for oi, d in enumerate(origins):
        # Persistence MASE of this origin anchors the pre-registered divergence rule.
        pers = inverse_scale(Persistence().predict(d.test_x, d.horizon), d)
        pers_mase = compute_metrics(d.test_y_raw, pers, d.train_raw)["MASE"]
        for seed in V4["seeds"]:
            if (name, oi, seed) in DONE_BLOCKS:
                n_reused += 1; continue
            set_seed(seed)
            rows, rels, shard = [], [], {}

            def record(mname, model, gate_val=np.nan, action="", extra=None):
                with torch.no_grad():
                    pr = inverse_scale(model(torch.as_tensor(d.test_x).to(DEVICE))
                                       .cpu().numpy(), d)
                    pc = inverse_scale(model(torch.as_tensor(d.val_x).to(DEVICE))
                                       .cpu().numpy(), d)
                met = compute_metrics(d.test_y_raw, pr, d.train_raw)
                rows.append(dict(dataset=name, origin=oi, seed=seed, model=mname,
                                 gate=float(gate_val) if np.isfinite(gate_val) else np.nan,
                                 action=action, cal_MASE=calibration_mase(pc, d),
                                 persistence_MASE=pers_mase,
                                 diverged=bool(met["MASE"] > 3 * pers_mase),
                                 **(extra or {}), **met))
                shard[mname] = per_window_mae(d.test_y_raw, pr).astype(np.float32)

            # (1) Standalone students: trained once, reused by every rejecting policy.
            ns_models, ns_cal = {}, {}
            for s_name in V4["students"]:
                lr_ = v4_select_lr(s_name, d, seed)
                m, _ = train_distilled_student(build_model(s_name, d.horizon), d, seed,
                                               None, "NS", lr=lr_)
                ns_models[s_name] = m; n_trained += 1
                record(f"{s_name}_NS", m)
                ns_cal[s_name] = rows[-1]["cal_MASE"]

            # (2) Teachers fitted once per (origin, seed); predictions reused by all arms.
            for t_name in V4_TEACHERS:
                teacher = GEN_TEACHER_FACTORY[t_name](seed).fit(
                    d.train_x, d.train_y, d.val_x, d.val_y)
                r_leg, mt_leg, mn_leg = teacher_reliability(teacher, d)   # legacy score (floored)
                for s_name in V4["students"]:
                    lr_ = v4_select_lr(s_name, d, seed)
                    r_stu, mt_stu = student_relative_reliability(teacher, d, ns_cal[s_name])
                    g_soft, act = gate_policy(r_stu)
                    g_hard = hard_reject_sr(r_stu)
                    rels.append(dict(dataset=name, origin=oi, seed=seed, teacher=t_name,
                                     student=s_name, r_stu=r_stu, r_legacy=r_leg,
                                     teacher_cal_MASE=mt_stu, ns_cal_MASE=ns_cal[s_name],
                                     g_soft=g_soft, action=act, g_hard=g_hard))
                    arms = {"NaiveKD":       dict(kind="Emulator", r_override=None),
                            "SoftSR":        dict(kind="Adaptive", r_override=g_soft),
                            "HardRejectSR":  dict(kind="Adaptive", r_override=g_hard),
                            "LegacyPersist": dict(kind="Adaptive", r_override=r_leg)}
                    for pol, kw in arms.items():
                        m, g_eff = train_distilled_student(
                            build_model(s_name, d.horizon), d, seed, teacher,
                            lr=lr_, ns_model=ns_models[s_name], **kw)
                        if m is ns_models[s_name]: n_reused += 1
                        else:                      n_trained += 1
                        record(f"{s_name}_{pol}[{t_name}]", m, g_eff,
                               act if pol == "SoftSR" else "",
                               extra=dict(teacher=t_name, student=s_name, policy=pol))
            # (3) Checkpoint this (dataset, origin, seed) block atomically.
            pd.DataFrame(rows).to_csv(V4_RUNS_CSV, mode="a", index=False,
                                      header=not os.path.exists(V4_RUNS_CSV))
            pd.DataFrame(rels).to_csv(V4_REL_CSV, mode="a", index=False,
                                      header=not os.path.exists(V4_REL_CSV))
            np.savez_compressed(_v4_shard_path(name, oi, seed), **shard)
            print(f"  o{oi} seed {seed} done | trained {n_trained} reused {n_reused} "
                  f"| elapsed {elapsed()}")

pd.DataFrame(ORIGIN_BOUNDS).drop_duplicates().to_csv(
    f"{RESULTS_DIR}/origin_boundaries.csv", index=False)
v4_runs = pd.read_csv(V4_RUNS_CSV).drop_duplicates(
    subset=["dataset", "origin", "seed", "model"], keep="last")
v4_rel = pd.read_csv(V4_REL_CSV).drop_duplicates(
    subset=["dataset", "origin", "seed", "teacher", "student"], keep="last")
print(f"\n[v4] benchmark complete in {(time.time()-t0_v4)/60:.1f} min | "
      f"runs={len(v4_runs)} | reliability rows={len(v4_rel)}")


checkpoint: 0 (dataset, origin, seed) blocks already complete

=== [v4] ETTh1: 5 origins | elapsed 230.6 min ===
    [v4 lr-select] PatchTST on ETTh1: 0.0003
  o0 seed 42 done | trained 24 reused 15 | elapsed 234.0 min
  o0 seed 123 done | trained 48 reused 30 | elapsed 236.8 min
  o0 seed 2024 done | trained 72 reused 45 | elapsed 239.9 min
  o1 seed 42 done | trained 96 reused 60 | elapsed 243.4 min
  o1 seed 123 done | trained 120 reused 75 | elapsed 247.0 min
  o1 seed 2024 done | trained 144 reused 90 | elapsed 250.6 min
  o2 seed 42 done | trained 168 reused 105 | elapsed 253.9 min
  o2 seed 123 done | trained 192 reused 120 | elapsed 257.3 min
  o2 seed 2024 done | trained 216 reused 135 | elapsed 260.5 min
  o3 seed 42 done | trained 240 reused 150 | elapsed 264.6 min
  o3 seed 123 done | trained 264 reused 165 | elapsed 269.4 min
  o3 seed 2024 done | trained 288 reused 180 | elapsed 274.0 min
  o4 seed 42 done | trained 312 reused 195 | elapsed 280.7 min
  o4 seed 123 done | 

## Part III-bis (post-hoc) — derived comparators, at zero training cost (R5)

Three decision rules a sceptical reviewer would propose are computable from the recorded runs:
`ValSelect` (choose NS or NaiveKD by calibration MASE, report the chosen arm's test error),
`HardRejectPersist` (the persistence-relative *hard* rule: NaiveKD iff the teacher beats
persistence on calibration, else NS) and `Oracle` (a-posteriori best of NS / NaiveKD — a
headroom bound, never a deployable method). Their per-window error shards are copies of the
selected arm's shard, so every statistical procedure below treats them identically.

In [22]:

# ============ v4 POST-HOC COMPARATORS: ValSelect / HardRejectPersist / Oracle =
def _pw4_load(ds_base, origin, seed):
    p = _v4_shard_path(ds_base, origin, seed)
    return dict(np.load(p)) if os.path.exists(p) else {}

DERIVED_ROWS = []
for (name, oi, seed), blk in v4_runs.groupby(["dataset", "origin", "seed"]):
    shard = _pw4_load(name, oi, seed)
    if not shard: continue
    new_shard = {}
    by_model = blk.set_index("model")
    for t_name in V4_TEACHERS:
        rel_rows = v4_rel[(v4_rel.dataset == name) & (v4_rel.origin == oi)
                          & (v4_rel.seed == seed) & (v4_rel.teacher == t_name)]
        for s_name in V4["students"]:
            ns_n, kd_n = f"{s_name}_NS", f"{s_name}_NaiveKD[{t_name}]"
            if ns_n not in by_model.index or kd_n not in by_model.index: continue
            ns_r, kd_r = by_model.loc[ns_n], by_model.loc[kd_n]
            rr = rel_rows[rel_rows.student == s_name]
            r_leg = float(rr.r_legacy.iloc[0]) if len(rr) else np.nan
            picks = {
                # ValSelect: model selection on the calibration block only.
                "ValSelect": (kd_n if kd_r.cal_MASE < ns_r.cal_MASE else ns_n),
                # HardRejectPersist: r_legacy = 1 iff teacher beats persistence.
                "HardRejectPersist": (kd_n if r_leg >= 1.0 - 1e-9 else ns_n),
                # Oracle: a-posteriori best on TEST - headroom only (declared as such).
                "Oracle": (kd_n if kd_r.MASE < ns_r.MASE else ns_n),
            }
            for pol, src in picks.items():
                r = by_model.loc[src].to_dict()
                mname = f"{s_name}_{pol}[{t_name}]"
                DERIVED_ROWS.append(dict(dataset=name, origin=oi, seed=seed, model=mname,
                                         teacher=t_name, student=s_name, policy=pol,
                                         picked=src, **{k: r[k] for k in
                                         ("MASE", "MAE", "RMSE", "cal_MASE",
                                          "persistence_MASE", "diverged")}))
                if src in shard: new_shard[mname] = shard[src]
    if new_shard:
        shard.update(new_shard)
        np.savez_compressed(_v4_shard_path(name, oi, seed), **shard)

v4_derived = pd.DataFrame(DERIVED_ROWS)
v4_derived.to_csv(f"{RESULTS_DIR}/v4_derived_policies.csv", index=False)
v4_all = pd.concat([v4_runs, v4_derived], ignore_index=True, sort=False)
ALL_POLICIES = V4_POLICIES + ["ValSelect", "HardRejectPersist", "Oracle"]
print(f"post-hoc comparators derived: {len(v4_derived)} rows | policies now: {ALL_POLICIES}")


post-hoc comparators derived: 2430 rows | policies now: ['NaiveKD', 'SoftSR', 'HardRejectSR', 'LegacyPersist', 'ValSelect', 'HardRejectPersist', 'Oracle']


In [23]:

# ============== v4 SUMMARY TABLE + Fig. 12 (descriptive layer) ================
# Regret is defined per (dataset, origin, seed, student, teacher, policy) as
# MASE_policy - MASE_NS on the SAME block: the standalone student is the safety
# comparator throughout (R2, R14). This cell is descriptive; confirmatory
# inference follows in the dedicated dependence-aware cell.

ns_key = v4_runs[v4_runs.model.str.endswith("_NS")].copy()
ns_key["student"] = ns_key.model.str.replace("_NS", "", regex=False)
ns_idx = ns_key.set_index(["dataset", "origin", "seed", "student"])["MASE"]

pol_rows = v4_all[v4_all.get("policy").notna()].copy() if "policy" in v4_all else v4_all.copy()
pol_rows = pol_rows[pol_rows.policy.isin(ALL_POLICIES)]
pol_rows["MASE_NS"] = [ns_idx.get((r.dataset, r.origin, r.seed, r.student), np.nan)
                       for r in pol_rows.itertuples()]
pol_rows["regret"] = pol_rows["MASE"] - pol_rows["MASE_NS"]
pol_rows["rel_regret"] = pol_rows["regret"] / pol_rows["MASE_NS"]
pol_rows.to_csv(f"{RESULTS_DIR}/v4_regret_long.csv", index=False)

# Origin-level aggregation: seeds averaged WITHIN the unit for display; the
# confirmatory cell keeps seeds as a modelled level.
unit = (pol_rows.groupby(["dataset", "origin", "student", "teacher", "policy"])
        [["MASE", "MASE_NS", "regret", "rel_regret"]].mean().reset_index())
unit.to_csv(f"{RESULTS_DIR}/v4_regret_by_origin.csv", index=False)

summ = (unit.groupby(["student", "teacher", "policy"])
        .agg(mean_regret=("regret", "mean"), median_regret=("regret", "median"),
             harm_rate=("regret", lambda v: float(np.mean(v > EPSILON_HARM))),
             n_units=("regret", "size")).round(4).reset_index())
summ.to_csv(f"{RESULTS_DIR}/v4_summary.csv", index=False)
print("=== v4 summary: regret vs standalone student (units = dataset x origin) ===")
print(summ.to_string(index=False))

# Fig. 12 (v4): regret distributions per policy, pooled over teachers, one panel
# per student. The zero line is the standalone student.
show_pols = ["NaiveKD", "LegacyPersist", "HardRejectSR", "SoftSR", "ValSelect"]
fig, axes = plt.subplots(1, len(V4["students"]), figsize=(FIG_W2, 2.6), sharey=True)
axes = np.atleast_1d(axes)
for ax, s_name in zip(axes, V4["students"]):
    sub = unit[unit.student == s_name]
    data = [sub[sub.policy == p]["regret"].dropna().values for p in show_pols]
    bp = ax.boxplot(data, tick_labels=show_pols, showfliers=True, patch_artist=True,
                    flierprops=dict(marker=".", ms=2))
    for pt in bp["boxes"]: pt.set_facecolor("#B8CCE4")
    ax.axhline(0, color="#C44E52", lw=0.8, ls="--")
    ax.set_title(f"student: {s_name}"); ax.tick_params(axis="x", rotation=35)
    ax.set_yscale("symlog", linthresh=0.5)
axes[0].set_ylabel("regret vs standalone (MASE, symlog)")
fig.suptitle("Fig. 12 (v4) - Negative-transfer regret by gate policy, dataset x origin units", y=1.04)
fig.tight_layout(); savefig(fig, "fig12_v4_regret_by_policy")


=== v4 summary: regret vs standalone student (units = dataset x origin) ===
 student      teacher            policy  mean_regret  median_regret  harm_rate  n_units
 DLinear ClassicalELM HardRejectPersist       0.0191         0.0000     0.4000       30
 DLinear ClassicalELM      HardRejectSR      -0.0008         0.0000     0.0000       30
 DLinear ClassicalELM     LegacyPersist       0.0454         0.0190     0.6667       30
 DLinear ClassicalELM           NaiveKD       0.0918         0.0170     0.7333       30
 DLinear ClassicalELM            Oracle      -0.0064         0.0000     0.0000       30
 DLinear ClassicalELM            SoftSR       0.0120         0.0021     0.5667       30
 DLinear ClassicalELM         ValSelect      -0.0027         0.0000     0.0667       30
 DLinear         QELM HardRejectPersist       0.0138         0.0000     0.0667       30
 DLinear         QELM      HardRejectSR      -0.0009         0.0000     0.0000       30
 DLinear         QELM     LegacyPersist     

## Part III-bis (confirmatory) — dependence-aware inference (R8)

**Experimental unit**: the (dataset, origin) block — test blocks are non-overlapping by
construction, so blocks are the defensible replication units; overlapping windows within a
block are repeated measures and never enter a test as independent observations.

**Primary tests** (family H1, Holm-corrected): one-sided Wilcoxon signed-rank over
origin-block regrets, `SoftSR < NaiveKD` and `SoftSR < LegacyPersist`, per (student, teacher).
**Sensitivity**: (i) per-origin Diebold-Mariano on seed-averaged per-window loss differentials
with HAC (Newey-West) variance truncated at h−1; (ii) moving-block bootstrap
(block length = W + h) reproducing the CIs; (iii) effective-sample-size diagnostics from the
loss-differential autocorrelation — the direct answer to the pseudoreplication objection.
**H2/H3**: non-inferiority via bootstrap upper CIs against the pre-registered 5% margin.

In [24]:

# =========== v4 CONFIRMATORY INFERENCE: origin blocks, DM, bootstrap ==========
def diebold_mariano(e_a, e_b, h):
    # DM statistic on one origin loss differential d_t = e_a - e_t of per-window
    # MAEs; HAC (Newey-West) long-run variance truncated at h-1, the standard
    # choice for h-step-ahead forecasts (Diebold & Mariano 1995).
    d = np.asarray(e_a, float) - np.asarray(e_b, float)
    d = d[np.isfinite(d)]; n = len(d)
    if n < 30: return np.nan, np.nan, n
    dbar = float(d.mean()); dc = d - dbar
    s = float(np.mean(dc * dc))
    for k in range(1, min(h, n - 1)):
        s += 2.0 * (1.0 - k / h) * float(np.mean(dc[k:] * dc[:-k]))
    if s <= 0: return np.nan, np.nan, n
    dm = dbar / np.sqrt(s / n)
    p = float(2 * stats.t.sf(abs(dm), df=n - 1))
    return float(dm), p, n

def ess_ratio(e_a, e_b, max_lag=None):
    # Effective-sample-size ratio n_eff / n of the loss differential, from the
    # truncated autocorrelation sum: quantifies how much the raw window count
    # overstates the information content (R8 evidence, Fig. 7 of the spec).
    d = np.asarray(e_a, float) - np.asarray(e_b, float)
    d = d[np.isfinite(d)]; n = len(d)
    if n < 50: return np.nan
    max_lag = max_lag or min(WINDOW + MAIN_HORIZON, n // 4)
    dc = d - d.mean(); v = float(np.mean(dc * dc)) + 1e-12
    rho_sum = 0.0
    for k in range(1, max_lag):
        r = float(np.mean(dc[k:] * dc[:-k])) / v
        if r < 0.05: break
        rho_sum += r
    return float(1.0 / (1.0 + 2.0 * rho_sum))

def moving_block_bootstrap_mean(d_vec, block, n_boot, seed=0):
    # Moving-block bootstrap of the mean of a serially dependent vector.
    d_vec = np.asarray(d_vec, float); d_vec = d_vec[np.isfinite(d_vec)]
    n = len(d_vec)
    if n < 2 * block: return np.array([])
    rng = np.random.default_rng(seed)
    n_blocks = int(np.ceil(n / block))
    starts = rng.integers(0, n - block + 1, size=(n_boot, n_blocks))
    out = np.empty(n_boot)
    for i in range(n_boot):
        take = np.concatenate([d_vec[s:s + block] for s in starts[i]])[:n]
        out[i] = take.mean()
    return out

def pw4_vec(name, oi, model, seeds):
    # Seed-averaged per-window MAE vector loaded from the persisted shards.
    vs = []
    for s in seeds:
        sh = _pw4_load(name, oi, s)
        if model in sh: vs.append(sh[model])
    if not vs: return None
    L = min(len(v) for v in vs)
    return np.nanmean(np.stack([v[:L] for v in vs]), axis=0)

# ---- (A) H1 primary: Wilcoxon over origin blocks, Holm within the family ----
H1_PAIRS = [("SoftSR", "NaiveKD"), ("SoftSR", "LegacyPersist")]
h1_rows = []
for s_name in V4["students"]:
    for t_name in V4_TEACHERS:
        cell = unit[(unit.student == s_name) & (unit.teacher == t_name)]
        piv = cell.pivot_table(index=["dataset", "origin"], columns="policy", values="regret")
        for a, b in H1_PAIRS:
            if a not in piv or b not in piv: continue
            m = piv[[a, b]].dropna()
            if len(m) < 5: continue   # one-sided Wilcoxon needs n >= 5 to reject at 0.05
            try:
                p = float(stats.wilcoxon(m[a], m[b], alternative="less",
                                         zero_method="wilcox").pvalue)
            except ValueError:
                p = 1.0
            dlt, eff = cliffs_delta(m[a].values, m[b].values)
            h1_rows.append(dict(family="H1", student=s_name, teacher=t_name,
                                comparison=f"{a} < {b}", n_blocks=len(m), p_value=p,
                                cliffs_delta=dlt, effect=eff,
                                mean_diff=float((m[a] - m[b]).mean())))
H1_COLUMNS = ["family", "student", "teacher", "comparison", "n_blocks", "p_value",
              "cliffs_delta", "effect", "mean_diff", "p_holm", "significant"]
h1 = pd.DataFrame(h1_rows)
if len(h1):
    h1["p_holm"] = holm(h1.p_value.values); h1["significant"] = h1.p_holm < STAT_ALPHA
else:
    h1 = pd.DataFrame(columns=H1_COLUMNS)   # stable schema for downstream readers

# ---- (B) Sensitivity: per-origin DM + block bootstrap + ESS diagnostics -----
dm_rows = []
for s_name in V4["students"]:
    for t_name in V4_TEACHERS:
        for (name, oi), _ in unit[(unit.student == s_name) & (unit.teacher == t_name)] \
                .groupby(["dataset", "origin"]):
            ea = pw4_vec(name, oi, f"{s_name}_SoftSR[{t_name}]", V4["seeds"])
            for ref_pol in ("NaiveKD", "LegacyPersist"):
                eb = pw4_vec(name, oi, f"{s_name}_{ref_pol}[{t_name}]", V4["seeds"])
                if ea is None or eb is None: continue
                L = min(len(ea), len(eb))
                dm, p, n = diebold_mariano(ea[:L], eb[:L], MAIN_HORIZON)
                boot = moving_block_bootstrap_mean(ea[:L] - eb[:L],
                                                   block=WINDOW + MAIN_HORIZON,
                                                   n_boot=min(CFG["bootstrap"], 2000))
                ci = (np.percentile(boot, [2.5, 97.5]).tolist() if len(boot) else [np.nan] * 2)
                dm_rows.append(dict(student=s_name, teacher=t_name, dataset=name, origin=oi,
                                    reference=ref_pol, n_windows=n, dm_stat=dm, dm_p=p,
                                    boot_lo=ci[0], boot_hi=ci[1],
                                    ess_ratio=ess_ratio(ea[:L], eb[:L])))
dm_df = pd.DataFrame(dm_rows)

# ---- (C) H2 / H3: non-inferiority against the pre-registered 5% margin ------
def block_ci_upper(vals, n_boot=None, seed=0):
    # Percentile bootstrap upper 95% bound over dataset x origin units.
    vals = np.asarray(vals, float); vals = vals[np.isfinite(vals)]
    if len(vals) < 4: return np.nan
    rng = np.random.default_rng(seed); n_boot = n_boot or CFG["bootstrap"]
    return float(np.percentile(
        [rng.choice(vals, len(vals), True).mean() for _ in range(n_boot)], 95))

h23_rows = []
for s_name in V4["students"]:
    cell = unit[unit.student == s_name]
    piv = cell.pivot_table(index=["dataset", "origin", "teacher"],
                           columns="policy", values="rel_regret")
    # H2: SoftSR non-inferior to HardRejectSR on mean relative regret.
    if {"SoftSR", "HardRejectSR"} <= set(piv.columns):
        m = piv[["SoftSR", "HardRejectSR"]].dropna()
        up = block_ci_upper((m.SoftSR - m.HardRejectSR).values)
        h23_rows.append(dict(family="H2", student=s_name, n_units=len(m),
                             estimate=float((m.SoftSR - m.HardRejectSR).mean()),
                             upper95=up, margin=NONINF_MARGIN,
                             passed=bool(np.isfinite(up) and up < NONINF_MARGIN)))
    # H3: on gate-accepted cells, SoftSR vs best(NS, NaiveKD).
    acc = v4_rel[(v4_rel.student == s_name) & (v4_rel.action == "accept")]         [["dataset", "origin", "teacher"]].drop_duplicates()
    if len(acc):
        cell_a = cell.merge(acc, on=["dataset", "origin", "teacher"])
        pa = cell_a.pivot_table(index=["dataset", "origin", "teacher"],
                                columns="policy", values="rel_regret")
        if {"SoftSR", "NaiveKD"} <= set(pa.columns):
            best_ref = np.minimum(pa["NaiveKD"].fillna(np.inf), 0.0)  # NS rel_regret = 0
            m = (pa["SoftSR"] - best_ref).dropna()
            up = block_ci_upper(m.values)
            h23_rows.append(dict(family="H3", student=s_name, n_units=len(m),
                                 estimate=float(m.mean()), upper95=up, margin=NONINF_MARGIN,
                                 passed=bool(np.isfinite(up) and up < NONINF_MARGIN)))
h23 = pd.DataFrame(h23_rows)

h1.to_csv(f"{RESULTS_DIR}/v4_confirmatory_H1.csv", index=False)
dm_df.to_csv(f"{RESULTS_DIR}/v4_sensitivity_DM.csv", index=False)
h23.to_csv(f"{RESULTS_DIR}/v4_confirmatory_H2_H3.csv", index=False)
print("=== H1 (origin-block Wilcoxon, Holm) ===")
print(h1.round(4).to_string(index=False) if len(h1) else "no H1 rows")
print("\n=== H2 / H3 (non-inferiority, bootstrap upper CI vs 5% margin) ===")
print(h23.round(4).to_string(index=False) if len(h23) else "no H2/H3 rows")
if len(dm_df):
    agree = dm_df.assign(sig=dm_df.dm_p < STAT_ALPHA, neg=dm_df.dm_stat < 0)
    print(f"\nDM sensitivity: {int((agree.sig & agree.neg).sum())}/{len(dm_df)} origin-level "
          f"comparisons significant in the protective direction | "
          f"median ESS ratio = {dm_df.ess_ratio.median():.3f} "
          f"(raw window counts overstate information by ~{1/max(dm_df.ess_ratio.median(),1e-3):.0f}x)")


=== H1 (origin-block Wilcoxon, Holm) ===
family  student      teacher             comparison  n_blocks  p_value  cliffs_delta     effect  mean_diff  p_holm  significant
    H1  DLinear         QELM       SoftSR < NaiveKD        30   0.0000       -0.6933      large    -1.6517  0.0000         True
    H1  DLinear         QELM SoftSR < LegacyPersist        30   0.0001       -0.4333     medium    -0.3563  0.0006         True
    H1  DLinear    QELM-NISQ       SoftSR < NaiveKD        30   0.0000       -0.6933      large    -1.8246  0.0000         True
    H1  DLinear    QELM-NISQ SoftSR < LegacyPersist        30   0.0000       -0.4333     medium    -0.3863  0.0002         True
    H1  DLinear ClassicalELM       SoftSR < NaiveKD        30   0.0002       -0.2956      small    -0.0798  0.0015         True
    H1  DLinear ClassicalELM SoftSR < LegacyPersist        30   0.0007       -0.2189      small    -0.0334  0.0033         True
    H1 PatchTST         QELM       SoftSR < NaiveKD        30  

## Part III-bis (endpoints) — downside-risk specification (R14)

Safety is reported as risk control relative to the standalone student, not as average
accuracy: negative-transfer rate with bootstrap upper 95% CI, regret tail (P95 / P99 / CVaR95),
divergence probability under the pre-registered rule, benefit retention against the oracle on
teacher-useful cells, and a post-hoc risk-coverage curve obtained by sweeping the acceptance
threshold τ of a hard student-relative gate over the recorded `r_stu` values — no retraining
is required for the sweep because each τ selects between already-trained NS and NaiveKD arms.

In [25]:

# ================= v4 RISK ENDPOINTS + RISK-COVERAGE CURVE (R14) ==============
def cvar(v, q=0.95):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if len(v) < 5: return np.nan
    thr = np.percentile(v, 100 * q)
    tail = v[v >= thr]
    return float(tail.mean()) if len(tail) else np.nan

ep_rows = []
for s_name in V4["students"]:
    for pol in ALL_POLICIES:
        cell = unit[(unit.student == s_name) & (unit.policy == pol)]
        if not len(cell): continue
        reg = cell.regret.values
        div = v4_all[(v4_all.get("student") == s_name) & (v4_all.get("policy") == pol)]
        # Benefit retention on oracle-useful cells: (NS - pol) / (NS - Oracle).
        oc = unit[(unit.student == s_name) & (unit.policy == "Oracle")]             .set_index(["dataset", "origin", "teacher"])["MASE"]
        cc = cell.set_index(["dataset", "origin", "teacher"])
        useful = cc.join(oc.rename("MASE_oracle"), how="inner")
        useful = useful[useful.MASE_NS - useful.MASE_oracle > 1e-9]
        retention = float(((useful.MASE_NS - useful.MASE)
                           / (useful.MASE_NS - useful.MASE_oracle)).clip(-2, 1).mean())             if len(useful) else np.nan
        ep_rows.append(dict(
            student=s_name, policy=pol, n_units=len(cell),
            nt_rate=float(np.mean(reg > EPSILON_HARM)),
            nt_rate_upper95=block_ci_upper((reg > EPSILON_HARM).astype(float)),
            regret_mean=float(np.nanmean(reg)), regret_median=float(np.nanmedian(reg)),
            regret_P95=float(np.nanpercentile(reg, 95)),
            regret_P99=float(np.nanpercentile(reg, 99)), regret_CVaR95=cvar(reg),
            divergence_rate=float(div.diverged.mean()) if len(div) else np.nan,
            benefit_retention=retention))
endpoints = pd.DataFrame(ep_rows).round(4)
endpoints.to_csv(f"{RESULTS_DIR}/v4_risk_endpoints.csv", index=False)
print("=== v4 risk endpoints (unit = dataset x origin; comparator = standalone student) ===")
print(endpoints.to_string(index=False))

# ---- Risk-coverage: hard SR gate with swept acceptance threshold tau ---------
rc_rows = []
mase_pairs = (unit[unit.policy == "NaiveKD"]
              .set_index(["dataset", "origin", "teacher", "student"])[["MASE", "MASE_NS"]])
r_unit = (v4_rel.groupby(["dataset", "origin", "teacher", "student"])["r_stu"]
          .mean().rename("r_stu"))
rc = mase_pairs.join(r_unit, how="inner").reset_index()
for s_name in V4["students"]:
    sub = rc[rc.student == s_name]
    for tau in np.round(np.linspace(0.0, 1.0, 21), 3):
        sel = np.where(sub.r_stu >= tau, sub.MASE, sub.MASE_NS)   # accept KD iff r_stu >= tau
        reg = sel - sub.MASE_NS.values
        rc_rows.append(dict(student=s_name, tau=float(tau),
                            coverage=float(np.mean(sub.r_stu >= tau)),
                            harm_rate=float(np.mean(reg > EPSILON_HARM)),
                            mean_regret=float(np.mean(reg)), cvar95=cvar(reg)))
rc_df = pd.DataFrame(rc_rows)
rc_df.to_csv(f"{RESULTS_DIR}/v4_risk_coverage.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(FIG_W2, 2.4))
for s_name in V4["students"]:
    sub = rc_df[rc_df.student == s_name]
    axes[0].plot(sub.coverage, sub.harm_rate, marker=".", ms=3, label=s_name)
    axes[1].plot(sub.tau, sub.mean_regret, marker=".", ms=3, label=s_name)
axes[0].set_xlabel("coverage (fraction of teachers accepted)")
axes[0].set_ylabel("negative-transfer rate"); axes[0].set_title("risk-coverage (hard SR gate)")
axes[1].set_xlabel(r"acceptance threshold $\tau$"); axes[1].set_ylabel("mean regret (MASE)")
axes[1].axhline(0, color="#C44E52", lw=0.7, ls="--"); axes[1].set_title("regret vs threshold")
axes[0].legend()
fig.suptitle("Fig. 13 (v4) - Risk-coverage and threshold sensitivity of the student-relative gate", y=1.05)
fig.tight_layout(); savefig(fig, "fig13_v4_risk_coverage")


=== v4 risk endpoints (unit = dataset x origin; comparator = standalone student) ===
 student            policy  n_units  nt_rate  nt_rate_upper95  regret_mean  regret_median  regret_P95  regret_P99  regret_CVaR95  divergence_rate  benefit_retention
 DLinear           NaiveKD       90   0.8000           0.8667       1.1886         0.1921      5.8962      6.9329         6.7532           0.0000             0.3441
 DLinear            SoftSR       90   0.1889           0.2556       0.0033         0.0000      0.0339      0.1123         0.0710           0.0000             0.5297
 DLinear      HardRejectSR       90   0.0000           0.0000      -0.0008         0.0000      0.0000      0.0000         0.0000           0.0000             0.5198
 DLinear     LegacyPersist       90   0.6889           0.7667       0.2619         0.0581      1.2660      1.5095         1.4248           0.0000             0.5485
 DLinear         ValSelect       90   0.0222           0.0556      -0.0020         0.0000 

In [26]:
# --- 15-bis (e): gate functional-form sensitivity (reviewer point) ----------
# Same DLinear student, QELM teacher only, ablation seeds. Four monotone maps
# rho = MASE_teacher/MASE_naive -> r. Expectation (verified at small scale):
# in the deep-unreliable regime rho >> 1 all forms saturate and coincide.

GATE_FORMS = {
    "hyperbolic(paper)": lambda rho: float(np.clip(1.0/(1.0+max(0.0, rho-1.0)), 0.05, 1.0)),
    "sigmoid(k=5)":      lambda rho: float(np.clip(1.0/(1.0+np.exp(5.0*(rho-1.0))), 0.05, 1.0)),
    "linear":            lambda rho: float(np.clip(2.0-rho, 0.05, 1.0)),
    "hard-threshold":    lambda rho: 1.0 if rho <= 1.0 else 0.05,
}

form_rows = []
for name in AVAILABLE:
    d = prepare(name, RAW_SERIES[name], MAIN_HORIZON, CFG["max_points"])
    for seed in CFG["abl_seeds"]:
        set_seed(seed)
        teacher = make_teacher("QELM", seed).fit(d.train_x, d.train_y, d.val_x, d.val_y)
        _, mt, mn = teacher_reliability(teacher, d)
        rho = mt / max(mn, 1e-8)
        for form, fmap in GATE_FORMS.items():
            rv = fmap(rho)
            m, _ = train_distilled_student(DLinear(WINDOW, MAIN_HORIZON), d, seed,
                                           teacher, kind="Adaptive", r_override=rv)
            with torch.no_grad():
                pr = inverse_scale(m(torch.as_tensor(d.test_x).to(DEVICE)).cpu().numpy(), d)
            form_rows.append(dict(dataset=name, seed=seed, rho=rho, form=form, r=rv,
                                  MASE=compute_metrics(d.test_y_raw, pr, d.train_raw)["MASE"]))

form_df = pd.DataFrame(form_rows)
form_pivot = form_df.groupby(["dataset", "form"])["MASE"].mean().unstack().round(4)
print("=== gate functional-form sensitivity (Ada MASE, DLinear student, QELM teacher) ===")
print(form_pivot.to_string())
print("\nmean rho per dataset:")
print(form_df.groupby("dataset")["rho"].mean().round(2).to_string())
form_df.to_csv(f"{RESULTS_DIR}/generality_gate_forms.csv", index=False)
print(f"saved {RESULTS_DIR}/generality_gate_forms.csv")


=== gate functional-form sensitivity (Ada MASE, DLinear student, QELM teacher) ===
form          hard-threshold  hyperbolic(paper)  linear  sigmoid(k=5)
dataset                                                              
AAPL                  0.6074             0.6074  0.6074        0.6081
ETTh1                 1.8796             1.8805  1.8796        1.8796
ETTm2                 5.8396             7.4000  5.8396        5.8396
Energy                1.7773             1.9473  1.9185        1.7899
ExchangeRate          4.0141             3.9876  4.0141        4.0141
Jena                  7.6960             8.1933  7.6960        7.6960

mean rho per dataset:
dataset
AAPL             0.70
ETTh1           12.53
ETTm2            3.45
Energy           1.31
ExchangeRate    12.93
Jena             2.34
saved /kaggle/working/results/generality_gate_forms.csv


### 15-bis — Mapping to the manuscript and expected takeaways

**Where this goes in the paper.** A new subsection in Section IV
(*"Generality: strong students, classical teachers, and NISQ noise"*), with
Fig. 12 and one summary table; plus three sentences in the Introduction and
Discussion replacing claims that currently rest on the CeNN alone. It also
directly feeds the response letter:

| Reviewer objection | Cell(s) | Evidence produced |
|---|---|---|
| (B) gate only shown on a weak student | (b)+(c) | DLinear (top-tier baseline) is degraded by naive distillation from unreliable teachers and recovered by the gate, Holm-significant per-window Wilcoxon |
| (D/Q3) a classical unreliable teacher would behave the same | (a)+(c) | ClassicalELM shows the same dataset-dependent reliability profile; the gate protects identically → the mechanism is teacher-agnostic *including* non-quantum teachers |
| (C) statevector-only simulation is unrealistic | (a) | 1024-shot sampling noise + 1%/layer depolarizing: reliability landscape and protection unchanged |

**Known additional insight** (from the preliminary closed-form run): with a
teacher that is *reliable vs. persistence* but *weaker than the student*,
distillation still costs the student. `r_global` is persistence-relative, not
student-relative — this likely explains the C2 outlier (QRC/Energy, +44.6%)
and motivates a student-relative reliability score as future work; consider
adding one sentence to Section V-C.

**Honesty note.** If any cell above produces a result that contradicts the
expected pattern, report it as-is — the paper's credibility rests on the
pre-registered, transparent-reporting stance.


# Part IV — Publication figures
Every figure below is written to `figures/` **twice**: `*.png` (300 dpi, for the notebook) and
`*.pdf` (vector, Type-42 fonts, IEEE column widths — drop-in ready for the TAI LaTeX template).
Figure sizes: `FIG_W1 = 3.5 in` (single column), `FIG_W2 = 7.16 in` (double column).

## 17. Fig. 1 — Method overview (reliability-gated distillation)

In [27]:
fig, ax = plt.subplots(figsize=(FIG_W2, 2.9)); ax.axis("off")
def box(x, y, w, h, text, fc="#EAF0F8", ec="#4C72B0", fs=7, lw=0.9):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.012",
                                            fc=fc, ec=ec, lw=lw))
    ax.text(x+w/2, y+h/2, text, ha="center", va="center", fontsize=fs)
def arrow(x0, y0, x1, y1, color="#444444", ls="-", label=None, lfs=6):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=0.9, ls=ls))
    if label: ax.text((x0+x1)/2, (y0+y1)/2+0.03, label, ha="center", fontsize=lfs, color=color)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
box(0.01, 0.62, 0.15, 0.25, "Input window\n$x \\in \\mathbb{R}^{96}$", fc="#F5F5F5", ec="#8C8C8C")
box(0.23, 0.72, 0.22, 0.24, "Quantum teacher (sim.)\nQELM / VQC / QRC / QKRR\n6 qubits, statevector",
    fc="#F0EAF6", ec="#8172B3")
box(0.23, 0.10, 0.22, 0.24, "CeNN student\ncirculant 3-tap coupling\n+ forecast & emulation heads")
box(0.55, 0.72, 0.19, 0.24, "Observables\n$\\langle Z_i\\rangle,\\langle Z_iZ_{i+1}\\rangle,"
    "\\langle X_i\\rangle$", fc="#F0EAF6", ec="#8172B3")
box(0.55, 0.10, 0.19, 0.24, "Student forecast\n$\\hat{y}_{1..h}$")
box(0.80, 0.41, 0.19, 0.26, "Reliability gate\n$r_{\\mathrm{global}}$ (validation)\n"
    "$g_{\\mathrm{local}}$ ($\\pm4\\sigma$)", fc="#FDF3E7", ec="#DD8452")
arrow(0.16, 0.76, 0.23, 0.82); arrow(0.16, 0.70, 0.23, 0.26)
arrow(0.45, 0.84, 0.55, 0.84); arrow(0.645, 0.72, 0.645, 0.36, color="#8172B3", ls="--",
                                     label="distillation loss (gated)")
arrow(0.74, 0.80, 0.83, 0.67, color="#DD8452"); arrow(0.74, 0.22, 0.83, 0.43, color="#DD8452")
arrow(0.895, 0.41, 0.66, 0.30, color="#DD8452", ls=":")
ax.text(0.5, 0.005, "Gate closes when validation reliability is low "
        r"$\Rightarrow$ negative transfer is bounded by construction", ha="center", fontsize=6.5,
        style="italic", color="#555555")
savefig(fig, "fig1_architecture")

[fig] fig1_architecture.pdf / .png saved


## 18. Fig. 2 — Teacher reliability map (the failure landscape)

In [28]:
rel_p = reliability_df.pivot_table(index="teacher", columns="dataset",
                                   values="r_global", aggfunc="mean").reindex(TEACHERS)
fig, ax = plt.subplots(figsize=(FIG_W1, 2.1))
im = ax.imshow(rel_p.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(rel_p.shape[1])); ax.set_xticklabels(rel_p.columns, rotation=40, ha="right")
ax.set_yticks(range(rel_p.shape[0])); ax.set_yticklabels(rel_p.index)
for i in range(rel_p.shape[0]):
    for j in range(rel_p.shape[1]):
        v = rel_p.values[i, j]
        if np.isfinite(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6,
                    color="black" if 0.25 < v < 0.85 else "white")
cb = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02); cb.set_label(r"$r_{\mathrm{global}}$")
ax.set_title("Teacher validation reliability", fontsize=8)
savefig(fig, "fig2_teacher_reliability")
print(rel_p.round(3))

[fig] fig2_teacher_reliability.pdf / .png saved
dataset  AAPL  ETTh1  ETTm2  Energy  ExchangeRate   Jena
teacher                                                 
QELM      1.0  0.103  0.290   0.751         0.085  0.436
VQC       1.0  0.129  0.470   0.817         0.094  0.521
QRC       1.0  0.241  0.669   0.968         0.060  0.498
QKRR      1.0  0.224  0.527   0.827         0.064  0.586


## 19. Fig. 3 — **Central result**: distillation gain vs. teacher reliability
Each point is one (teacher, dataset) pair. `y > 0` means the adaptive gate improved over naive
distillation. The prediction of C1 is that the largest gains concentrate at **low** reliability,
and that no point falls substantially below zero (bounded downside).

In [29]:
fig, ax = plt.subplots(figsize=(FIG_W1, 2.5))
for t in TEACHERS:
    sub = gating_df[gating_df.teacher == t]
    gain = sub["CeNN_Emulator"] - sub["CeNN_Adaptive"]
    ax.scatter(sub["r_global"], gain, s=18, color=TEACHER_COLORS[t], label=t, zorder=3,
               edgecolors="white", linewidths=0.4)
ax.axhline(0, color="#888888", lw=0.7)
ax.axvspan(0, UNRELIABLE_R, color="#DD8452", alpha=0.07)
ax.axvspan(RELIABLE_R, 1.0, color="#55A868", alpha=0.07)
ax.text(UNRELIABLE_R/2, ax.get_ylim()[1]*0.92, "unreliable", ha="center", fontsize=6, color="#B35C1E")
ax.set_xlabel(r"teacher reliability $r_{\mathrm{global}}$ (validation)")
ax.set_ylabel(r"MASE gain of gating" + "\n" + r"(Emulator $-$ Adaptive)")
ax.set_yscale("symlog", linthresh=1); ax.legend(ncols=2, fontsize=6, frameon=False)
savefig(fig, "fig3_gain_vs_reliability")

[fig] fig3_gain_vs_reliability.pdf / .png saved


## 20. Fig. 4 — NS / Emulator / Adaptive per teacher, split by reliability regime

In [30]:
reg_order = ["reliable teacher", "intermediate", "unreliable teacher"]
agg = (gating_df.assign(regime=pd.Categorical(gating_df.regime, reg_order))
       .groupby(["regime", "teacher"], observed=True)[["CeNN_NS", "CeNN_Emulator", "CeNN_Adaptive"]]
       .median())
fig, axes = plt.subplots(1, 3, figsize=(FIG_W2, 2.2), sharey=False)
for ax, reg in zip(axes, reg_order):
    if reg not in agg.index.get_level_values(0):
        ax.set_axis_off(); ax.set_title(f"{reg}\n(no cases)", fontsize=7); continue
    sub = agg.loc[reg].reindex(TEACHERS).dropna(how="all")
    x = np.arange(len(sub)); w = 0.27
    ax.bar(x-w, sub["CeNN_NS"], w, label="NS (no distill.)", color="#4C72B0")
    ax.bar(x,   sub["CeNN_Emulator"], w, label="Emulator (always-on)", color="#C44E52")
    ax.bar(x+w, sub["CeNN_Adaptive"], w, label="Adaptive (gated)", color="#55A868")
    ax.set_xticks(x); ax.set_xticklabels(sub.index, fontsize=6.5)
    ax.set_title(reg, fontsize=7.5); ax.set_yscale("log")
axes[0].set_ylabel("median MASE (log)")
axes[0].legend(fontsize=6, frameon=False, loc="upper left")
savefig(fig, "fig4_regime_bars")

[fig] fig4_regime_bars.pdf / .png saved


## 21. Fig. 5 — Critical-difference diagram (Friedman/Nemenyi, dataset×horizon blocks)

In [31]:
def cd_diagram(avg_ranks, cd, fname, title=""):
    r = avg_ranks.sort_values(); k = len(r)
    lo, hi = np.floor(r.min()), np.ceil(r.max())
    fig, ax = plt.subplots(figsize=(FIG_W2, 0.42*k/2 + 1.2)); ax.set_axis_off()
    ax.set_xlim(lo-0.4, hi+0.4); ax.set_ylim(-0.7*(k/2)-1.6, 1.9)
    for x in np.arange(lo, hi+1): ax.plot([x, x], [0, 0.12], color="black", lw=0.7)
    ax.plot([lo, hi], [0, 0], color="black", lw=0.9)
    for x in np.arange(lo, hi+1): ax.text(x, 0.28, f"{int(x)}", ha="center", fontsize=7)
    ax.plot([lo, lo+cd], [1.15, 1.15], color="black", lw=1.6)
    ax.text(lo+cd/2, 1.35, f"CD = {cd:.2f}", ha="center", fontsize=7)
    half = int(np.ceil(k/2)); step = 0.62
    for i, (m, val) in enumerate(r.items()):
        left = i < half
        row = i if left else (k-1-i)
        yl = -0.55 - row*step
        xt = lo-0.35 if left else hi+0.35
        ax.plot([val, val, xt], [0, yl, yl], color="#555555", lw=0.7)
        ax.text(xt, yl, f" {m} ({val:.2f}) " , fontsize=6.5,
                ha="right" if left else "left", va="center")
    # cliques: maximal groups within CD
    vals = r.values; names = r.index
    cliques, i = [], 0
    while i < k:
        j = i
        while j+1 < k and vals[j+1]-vals[i] <= cd: j += 1
        if j > i and not any(set(range(i, j+1)) <= set(c) for c in cliques):
            cliques.append(tuple(range(i, j+1)))
        i += 1
    for ci, c in enumerate(cliques):
        y = -0.18 - 0.10*ci
        ax.plot([vals[c[0]]-0.03, vals[c[-1]]+0.03], [y, y], color="black", lw=2.2,
                solid_capstyle="round")
    if title: ax.set_title(title, fontsize=8)
    savefig(fig, fname)

cd_diagram(avg_ranks_b, CD, "fig5_cd_diagram",
           f"Mean MASE ranks over {Nb} dataset×horizon blocks (k={k}); "
           f"Friedman p = {fried_p:.1e}")

[fig] fig5_cd_diagram.pdf / .png saved


## 22. Fig. 6 — MASE distributions across datasets × seeds (main horizon)

In [32]:
keep = [m for m in MODEL_ORDER if m in main_df.model.unique()]
fig, ax = plt.subplots(figsize=(FIG_W2, 2.8))
data = [main_df[main_df.model == m]["MASE"].dropna().clip(upper=1e4) for m in keep]
bp = ax.boxplot(data, tick_labels=keep, showfliers=True, patch_artist=True,
                flierprops=dict(marker=".", markersize=2.5, alpha=0.5),
                medianprops=dict(color="black", lw=1.0))
for patch, m in zip(bp["boxes"], keep):
    c = ("#55A868" if m.startswith("CeNN_Ada") else "#C44E52" if m.startswith("CeNN_Emu")
         else "#8172B3" if m in TEACHERS else "#4C72B0" if m == "CeNN_NS" else "#D9D9D9")
    patch.set_facecolor(c); patch.set_linewidth(0.6)
ax.axhline(1.0, color="#888888", lw=0.7, ls="--")
ax.text(0.4, 1.05, "naive level", fontsize=6, color="#666666")
ax.set_yscale("log"); ax.set_ylabel("MASE (log)"); plt.setp(ax.get_xticklabels(), rotation=75,
                                                            ha="right", fontsize=6)
savefig(fig, "fig6_mase_box")

[fig] fig6_mase_box.pdf / .png saved


## 23. Fig. 7 — Robustness across forecast horizons

In [33]:
mh_keep = ["Persistence", "DLinear", "CeNN_NS", "CeNN_Emu[QELM]", "CeNN_Ada[QELM]", "QELM"]
mh = (multih_df[multih_df.model.isin(mh_keep)]
      .groupby(["model", "horizon"])["MASE"].median().unstack())
styles = {"Persistence": ("#8C8C8C", ":"), "DLinear": ("#CCB974", "--"), "QELM": ("#8172B3", "-."),
          "CeNN_NS": ("#4C72B0", "-"), "CeNN_Emu[QELM]": ("#C44E52", "-"),
          "CeNN_Ada[QELM]": ("#55A868", "-")}
fig, ax = plt.subplots(figsize=(FIG_W1, 2.4))
for m in mh_keep:
    if m in mh.index:
        c, ls = styles[m]
        ax.plot(mh.columns, mh.loc[m], marker="o", ms=3, color=c, ls=ls, label=m)
ax.set_xticks(MULTI_HORIZONS); ax.set_xlabel("forecast horizon $h$")
ax.set_ylabel("median MASE"); ax.set_yscale("log")
ax.legend(fontsize=5.5, frameon=False, ncols=2)
savefig(fig, "fig7_multi_horizon")

[fig] fig7_multi_horizon.pdf / .png saved


## 24. Fig. 8 — Loss-term ablation (QELM-Adaptive)

In [34]:
ab = ablation_df.groupby(["dataset", "config"])["MASE"].median().unstack()
ab = ab[[c for c in ABL.keys() if c in ab.columns]]
fig, ax = plt.subplots(figsize=(FIG_W2, 2.3))
x = np.arange(len(ab.index)); nc = ab.shape[1]; w = 0.8/nc
cmap = plt.get_cmap("viridis")
for i, c in enumerate(ab.columns):
    ax.bar(x+(i-nc/2+0.5)*w, ab[c], w, label=c, color=cmap(i/(nc-1)))
ax.set_xticks(x); ax.set_xticklabels(ab.index, fontsize=6.5)
ax.set_yscale("log"); ax.set_ylabel("median MASE (log)")
ax.legend(fontsize=5.5, frameon=False, ncols=3)
savefig(fig, "fig8_ablation")
print(ab.round(3))

[fig] fig8_ablation.pdf / .png saved
config             F     F+S  F+S+St  F+S+St+D  F+S+St+D+Obs  Complete
dataset                                                               
AAPL           0.601   0.601   0.601     0.600         0.600     0.599
ETTh1          2.255   2.239   2.255     2.223         2.228     2.134
ETTm2          8.810   8.455   8.455    10.724        10.630    33.286
Energy         1.794   1.796   1.796     1.881         1.890     2.929
ExchangeRate  17.815  17.759  17.759    17.968        18.095    21.382
Jena           8.535   8.580   8.580     9.046         9.053    14.649


## 25. Fig. 9 — Emulation fidelity of the student observable head

In [35]:
obs = emulation_df[emulation_df.level == "observable"]
fid = obs.pivot_table(index="teacher", columns="dataset", values="emul_Pearson_r",
                      aggfunc="mean").reindex(TEACHERS)
fig, ax = plt.subplots(figsize=(FIG_W1, 2.1))
im = ax.imshow(fid.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(fid.shape[1])); ax.set_xticklabels(fid.columns, rotation=40, ha="right")
ax.set_yticks(range(fid.shape[0])); ax.set_yticklabels(fid.index)
for i in range(fid.shape[0]):
    for j in range(fid.shape[1]):
        v = fid.values[i, j]
        if np.isfinite(v): ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6,
                                   color="white" if v > 0.6 else "black")
cb = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cb.set_label("Pearson $r$ (observables)")
ax.set_title("Student $\\to$ teacher observable fidelity", fontsize=8)
savefig(fig, "fig9_fidelity")

[fig] fig9_fidelity.pdf / .png saved


## 26. Fig. 10 — Qualitative forecasts (first test windows, seed 0)

In [36]:
ex = np.load(f"{RESULTS_DIR}/prediction_examples.npz")
ex_keys = {tuple(k.split("|")): k for k in ex.files}
ds_show = [d_ for d_ in AVAILABLE if (d_, "__truth__") in ex_keys][:4]
mods = ["Persistence", "DLinear", "CeNN_NS", "CeNN_Ada[QELM]"]
colr = {"Persistence": "#8C8C8C", "DLinear": "#CCB974", "CeNN_NS": "#4C72B0",
        "CeNN_Ada[QELM]": "#55A868"}
fig, axes = plt.subplots(2, 2, figsize=(FIG_W2, 3.4))
for ax, ds_ in zip(axes.ravel(), ds_show):
    truth = ex[ex_keys[(ds_, "__truth__")]][0]
    ax.plot(truth, color="black", lw=1.2, label="ground truth")
    for m in mods:
        if (ds_, m) in ex_keys:
            ax.plot(ex[ex_keys[(ds_, m)]][0], color=colr[m], lw=0.9, label=m)
    ax.set_title(ds_, fontsize=7.5); ax.tick_params(labelsize=6)
axes[0, 0].legend(fontsize=5.5, frameon=False)
for ax in axes[-1]: ax.set_xlabel("step ahead", fontsize=7)
savefig(fig, "fig10_forecast_examples")

[fig] fig10_forecast_examples.pdf / .png saved


## 27. Fig. 11 — Gate telemetry during training (mechanism transparency)

In [37]:
gd = pd.read_csv(f"{RESULTS_DIR}/gate_dynamics.csv")   # Adaptive runs only, by construction
fig, axes = plt.subplots(1, 2, figsize=(FIG_W2, 2.2))
for t in TEACHERS:
    sub = gd[gd.teacher == t]
    if len(sub):
        g = sub.groupby("epoch")[["gate", "distill_weight"]].mean()
        axes[0].plot(g.index, g["gate"], color=TEACHER_COLORS[t], label=t)
        axes[1].plot(g.index, g["distill_weight"], color=TEACHER_COLORS[t], label=t)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("local gate open fraction"); axes[0].set_ylim(0, 1.02)
axes[1].set_xlabel("epoch"); axes[1].set_ylabel(r"effective distill. weight $\lambda\,r_{\mathrm{global}}$")
axes[0].legend(fontsize=6, frameon=False, ncols=2)
savefig(fig, "fig11_gate_dynamics")

[fig] fig11_gate_dynamics.pdf / .png saved


# Part V — Pre-registered claims: programmatic verification
The cell below re-derives every claim **from the saved artifacts** (not from in-memory state) and
prints a PASS/FAIL table. `claims_check.json` is exported so a reviewer can audit the exact
numbers. A FAIL does not crash the notebook — it is a scientific result to be reported honestly.

In [38]:

# ============= v4 PRE-REGISTERED CLAIMS: programmatic verification ============
# Every claim is re-derived from the persisted artifacts, never from in-memory
# state; a FAIL is a reportable finding, not an error. The legacy C1-C4 audit of
# the v3 study is preserved below the H-family for the comparison appendix.
claims = []
def claim(cid, statement, passed, evidence):
    claims.append(dict(claim=cid, statement=statement,
                       passed=bool(passed), evidence=evidence))

def _read_csv_safe(path):
    # Robust artifact read: an empty confirmatory family (e.g. smoke profile) is
    # a valid state and must yield an empty frame, not a crash.
    try:
        return pd.read_csv(path)
    except (FileNotFoundError, pd.errors.EmptyDataError):
        return pd.DataFrame()

h1_ = _read_csv_safe(f"{RESULTS_DIR}/v4_confirmatory_H1.csv")
h23_ = _read_csv_safe(f"{RESULTS_DIR}/v4_confirmatory_H2_H3.csv")
ep_ = _read_csv_safe(f"{RESULTS_DIR}/v4_risk_endpoints.csv")

# --- H1: harm control - SoftSR stochastically below NaiveKD and LegacyPersist
sig = h1_.groupby("comparison")["significant"].mean().to_dict() if len(h1_) else {}
claim("H1", "SoftSR regret < NaiveKD and < LegacyPersist over dataset x origin blocks (Holm)",
      len(h1_) > 0 and all(v >= 0.75 for v in sig.values()),
      dict(frac_significant_by_comparison={k: round(float(v), 3) for k, v in sig.items()},
           n_tests=int(len(h1_))))

# --- H2: incremental value of soft calibration over hard rejection (non-inferiority)
h2 = h23_[h23_.family == "H2"] if "family" in h23_ else pd.DataFrame()
claim("H2", "SoftSR non-inferior to HardRejectSR on mean relative regret (5% margin)",
      len(h2) > 0 and bool(h2.passed.all()),
      dict(per_student={r.student: dict(estimate=round(float(r.estimate), 4),
                                        upper95=round(float(r.upper95), 4))
                        for r in h2.itertuples()}))

# --- H3: non-inferiority on gate-accepted cells
h3 = h23_[h23_.family == "H3"] if "family" in h23_ else pd.DataFrame()
claim("H3", "On gate-accepted cells, SoftSR rel. regret vs best(NS, NaiveKD) within 5% margin",
      len(h3) > 0 and bool(h3.passed.all()),
      dict(per_student={r.student: dict(estimate=round(float(r.estimate), 4),
                                        upper95=round(float(r.upper95), 4))
                        for r in h3.itertuples()}))

# --- H4: generality - the H1 direction holds for every primary student
per_s = (h1_.groupby("student")["significant"].mean().round(3).to_dict()
         if len(h1_) else {})
claim("H4", "H1 protection direction holds for every primary student architecture",
      len(per_s) >= 2 and all(v >= 0.5 for v in per_s.values()),
      dict(frac_significant_per_student=per_s))

# --- S1 (specification 10.4): harm-rate bound for the primary gate
soft = ep_[ep_.policy == "SoftSR"] if "policy" in ep_ else pd.DataFrame()
claim("S1", "Upper 95% CI of the SoftSR negative-transfer rate reported per student "
            "(threshold to be fixed against the engineering tolerance in the manuscript)",
      bool(len(soft) > 0 and soft.nt_rate_upper95.notna().all()),
      dict(nt_rate_upper95_per_student={r.student: round(float(r.nt_rate_upper95), 3)
                                        for r in soft.itertuples()}))

claims_df = pd.DataFrame(claims)
json.dump(claims, open(f"{RESULTS_DIR}/claims_check_v4.json", "w"), indent=2)
print("=" * 72)
for c in claims:
    print(f"[{'PASS' if c['passed'] else 'FAIL'}] {c['claim']}: {c['statement']}")
    print(f"        evidence: {c['evidence']}")
print("=" * 72)
print("claims_check_v4.json written. FAILs are findings to report, never to hide.")

# ---------------- legacy C1-C4 audit (v3 study, comparison appendix) ----------
try:
    gat = pd.read_csv(f"{RESULTS_DIR}/adaptive_gating_study.csv")
    sta = pd.read_csv(f"{RESULTS_DIR}/statistical_comparison_full.csv")
    unrel = gat[gat.regime == "unreliable teacher"]
    c1 = float((unrel["CeNN_Adaptive"] < unrel["CeNN_Emulator"]).mean()) if len(unrel) else np.nan
    reli = gat[gat.regime == "reliable teacher"]
    ratio = (reli["CeNN_Adaptive"] / np.minimum(reli["CeNN_NS"], reli["CeNN_Emulator"]))         if len(reli) else pd.Series(dtype=float)
    legacy = dict(C1_frac_protected=round(c1, 3) if np.isfinite(c1) else None,
                  C2_worst_ratio=round(float(ratio.max()), 3) if len(ratio) else None)
    json.dump(legacy, open(f"{RESULTS_DIR}/claims_check_legacy.json", "w"), indent=2)
    print("legacy v3 audit:", legacy)
except FileNotFoundError:
    print("legacy v3 artifacts absent (Part II skipped): legacy audit not produced")


[PASS] H1: SoftSR regret < NaiveKD and < LegacyPersist over dataset x origin blocks (Holm)
        evidence: {'frac_significant_by_comparison': {'SoftSR < LegacyPersist': 0.778, 'SoftSR < NaiveKD': 0.778}, 'n_tests': 18}
[PASS] H2: SoftSR non-inferior to HardRejectSR on mean relative regret (5% margin)
        evidence: {'per_student': {'DLinear': {'estimate': 0.0005, 'upper95': 0.0012}, 'PatchTST': {'estimate': 0.0066, 'upper95': 0.0111}, 'N-HiTS': {'estimate': -0.0023, 'upper95': 0.0001}}}
[PASS] H3: On gate-accepted cells, SoftSR rel. regret vs best(NS, NaiveKD) within 5% margin
        evidence: {'per_student': {'DLinear': {'estimate': 0.0002, 'upper95': 0.0006}, 'PatchTST': {'estimate': 0.0099, 'upper95': 0.0193}, 'N-HiTS': {'estimate': 0.0038, 'upper95': 0.009}}}
[PASS] H4: H1 protection direction holds for every primary student architecture
        evidence: {'frac_significant_per_student': {'DLinear': 1.0, 'N-HiTS': 0.667, 'PatchTST': 0.667}}
[PASS] S1: Upper 95% CI of the Soft

## 29. Reproducibility bundle — checksums, environment, freeze

In [39]:
import hashlib, subprocess
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

sums = {}
for root in (RESULTS_DIR, FIG_DIR, DATA_DIR):
    for fn in sorted(os.listdir(root)):
        p = os.path.join(root, fn)
        if os.path.isfile(p): sums[os.path.relpath(p, WORKDIR)] = sha256(p)
json.dump(sums, open(f"{RESULTS_DIR}/checksums_sha256.json", "w"), indent=2)

frz = subprocess.run(["pip", "freeze"], capture_output=True, text=True).stdout
open(f"{RESULTS_DIR}/requirements_freeze.txt", "w").write(frz)

print(f"{len(sums)} artifacts hashed | figures:",
      len([f for f in os.listdir(FIG_DIR) if f.endswith('.pdf')]), "PDF /",
      len([f for f in os.listdir(FIG_DIR) if f.endswith('.png')]), "PNG")
print("total wall-clock:", elapsed())

66 artifacts hashed | figures: 14 PDF / 14 PNG
total wall-clock: 589.0 min


# Part VI — Mapping artifacts to the RINENG manuscript

| Manuscript element | Artifact |
|---|---|
| Table (origin boundaries, R7) | `results/origin_boundaries.csv` |
| Table (reliability scores, both references) | `results/v4_reliability.csv` |
| Table (primary safety/utility endpoints, R14) | `results/v4_risk_endpoints.csv` |
| Table (confirmatory H1, R8) | `results/v4_confirmatory_H1.csv` |
| Table (H2/H3 non-inferiority) | `results/v4_confirmatory_H2_H3.csv` |
| Table (DM + ESS sensitivity, R8) | `results/v4_sensitivity_DM.csv` |
| Table (nested objective selection, R13) | `results/objective_selection.csv` |
| Fig. regret by policy | `figures/fig12_v4_regret_by_policy.pdf` |
| Fig. risk-coverage | `figures/fig13_v4_risk_coverage.pdf` |
| Pre-registration snapshot | `results/config_v4.json` (frozen before training) |
| Claims audit | `results/claims_check_v4.json` + `claims_check_legacy.json` |
| Raw per-window errors (R15) | `results/pw_v4/*.npz` shards |
| Legacy v3 study (mechanistic section + appendix) | Parts I-II artifacts, unchanged |

Remaining manuscript-side items, outside this notebook (see the migration plan): engineering
system definition and FMEA (R1), critical SOTA review and novelty matrix (R3), dataset-portfolio
extension (R9), Zenodo DOI archive at submission (R15), Elsevier packaging (de-anonymised title
page, highlights ≤ 85 characters, CRediT, data statement).